In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL B0 - analysis setup: registry of 15 run payloads + audit ground truth
import os, json, glob, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

sys.path.insert(0, "/content/drive/MyDrive/medical_qa_hallucination/experiments")
from config import DATASET_FINAL, RESULTS_DIR, RESULTS_FINAL, EXPERIMENTS

import config
RESULTS_DIR   = Path(config.RESULTS_DIR)
RESULTS_FINAL = Path(config.RESULTS_FINAL)
AUDIT_DIR     = RESULTS_DIR.parent / "audit"        # read-only ground truth
ANALYSIS_DIR  = RESULTS_DIR.parent / "analysis"     # all analysis outputs go here
ANALYSIS_DIR.mkdir(exist_ok=True)
assert AUDIT_DIR.is_dir(), f"audit dir not found at {AUDIT_DIR}"

GROUPS    = ["reviewed20", "batch1", "batch2", "batch3", "batch4"]
ARM_CODES = {"coarse": "c", "atomic": "a",
             "atomic_gl_filter": "ag", "atomic_sec_filter": "as"}

def _questions(pl):
    for k in ("questions", "results", "items"):
        if isinstance(pl, dict) and k in pl:
            return pl[k]
    raise KeyError(f"cannot find question list in payload keys {list(pl)}")

def _pick_run_file(group, run):
    base = RESULTS_DIR if group == "reviewed20" else RESULTS_FINAL
    paths = sorted(glob.glob(str(base / f"results_{group}_run{run}*.json")))
    paths = [p for p in paths if os.path.basename(p).startswith(f"results_{group}_run{run}_")
             or os.path.basename(p) == f"results_{group}_run{run}.json"]
    assert paths, f"no run file for {group} run {run} in {base}"
    clean = [p for p in paths if "failed" not in os.path.basename(p).lower()]
    return (clean or paths)[-1]

RUNS, RUN_FILES, UID2GROUP = {}, {}, {}
for g in GROUPS:
    uid_sets = []
    for r in (1, 2, 3):
        p = _pick_run_file(g, r)
        with open(p) as f:
            pl = json.load(f)
        RUNS[f"{g}_run{r}"] = pl
        RUN_FILES[f"{g}_run{r}"] = p
        uid_sets.append({q["uid"] for q in _questions(pl)})
    assert uid_sets[0] == uid_sets[1] == uid_sets[2], f"{g}: uid sets differ across runs"
    for u in uid_sets[0]:
        UID2GROUP[u] = g

# cross-check against the locked dataset
ds_uids = set()
with open(config.DATASET_FINAL) as f:
    for line in f:
        ds_uids.add(json.loads(line)["uid"])
assert set(UID2GROUP) == ds_uids == set().union(*[
    {q["uid"] for q in _questions(RUNS[f"{g}_run1"])} for g in GROUPS]), "uid mismatch"
assert len(UID2GROUP) == 160

print("runs loaded:", len(RUNS), "(expect 15)")
print("uids:", len(UID2GROUP), "(expect 160)")
for g in GROUPS:
    print(f"  {g:12s} {sum(1 for u in UID2GROUP if UID2GROUP[u]==g):3d} uids")
print("audit dir:", AUDIT_DIR)

runs loaded: 15 (expect 15)
uids: 160 (expect 160)
  reviewed20    20 uids
  batch1        35 uids
  batch2        35 uids
  batch3        35 uids
  batch4        35 uids
audit dir: /content/drive/MyDrive/medical_qa_hallucination/experiments/audit


In [ ]:
# CELL B1 - Step 1: per-arm first-pass detection vs adjudicated ground truth
#
# What this cell does:
#   1. collects every first-pass (history[0]) refuted claim instance from the 15 run payloads
#      (schema: payload["results"] -> q["arms"][arm]["history"][0]["verified"])
#   2. assigns each instance to its audit row in audit_claims.csv, constrained by uid +
#      runs_flagged + arms_flagged + row capacity (n_instances), with a numeric guard so
#      claims that differ only in numbers can never be swapped
#   3. applies 7 manual overrides adjudicated via JSON claim ids (documented below)
#   4. validates the assignment against the audit (per-row counts, label totals)
#   5. produces the Step-1 tables (T1 arm x label, per-arm precision, T2 TH sensitivity)
#
# Matching notes (from diagnostics during development):
#   - audit claim_text is a paraphrase of the JSON claim; wording drifts across runs,
#     so text equality is impossible and unconstrained fuzzy matching is unsafe
#   - numbers are normalized ("3,000" -> 3000.0, "2.0" -> 2.0) before the guard
#   - constrained candidate sets are usually 1-2 rows, so low text similarity is expected
#     and harmless; forced/low-sim assignments are saved for provenance only

import re, difflib
from collections import defaultdict

# --- 1. collect first-pass refuted claim instances ---
_c0  = RUNS[sorted(RUNS, key=str)[0]]["results"][0]["arms"]["coarse"]["history"][0]["verified"][0]
K_TX = next(k for k in ["claim", "text", "claim_text", "statement"] if k in _c0)
K_V  = next(k for k in ["verdict", "label", "verification", "status"] if k in _c0)
print("claim fields:", sorted(_c0), "| using:", K_TX, "/", K_V)

instances, all_iter_refuted = [], 0
for tag, pl in RUNS.items():
    group, run = tag.rsplit("_run", 1)
    for q in pl["results"]:
        for arm, ares in q["arms"].items():
            for c in ares["history"][0]["verified"]:
                if str(c[K_V]).strip().lower() == "refuted":
                    instances.append(dict(group=group, run=int(run), arm=arm,
                                          uid=q["uid"], text=str(c[K_TX])))
            all_iter_refuted += sum(h.get("refuted", 0) for h in ares["history"])
df = pd.DataFrame(instances)
print(f"first-pass refuted instances: {len(df)} (expect 606) | "
      f"all-iteration occurrences (diagnostic): {all_iter_refuted}")

# --- 2. constrained assignment instance -> audit row ---
ac = pd.read_csv(os.path.join(AUDIT_DIR, "audit_claims.csv"))
_norm = lambda s: " ".join(str(s).lower().strip().strip('"').rstrip(" .;:").split())
ac["_n_parsed"] = ac["n_instances"].astype(str).str.split("/").str[0].astype(int)
ac["_runs"] = ac["runs_flagged"].map(lambda s: {int(x) for x in re.findall(r"\d+", str(s))})
ac["_arms"] = ac["arms_flagged"].map(
    lambda s: set(ARM_CODES.values()) if str(s).strip() == "4"
    else {x.strip() for x in str(s).split(",") if x.strip()})

def _nums(s):                                   # number multiset, format-robust
    return sorted(float(x) for x in re.findall(r"\d+(?:\.\d+)?", str(s).replace(",", "")))
_sim = lambda a, b: difflib.SequenceMatcher(None, _norm(a), _norm(b)).ratio()

rem = {i: int(r._n_parsed) for i, r in ac.iterrows()}
row_of, sim_of, forced = {}, {}, set()
unassigned = []
for n, inst in df.iterrows():
    cands = [i for i, r in ac.iterrows()
             if r.uid == inst.uid and inst.run in r._runs
             and ARM_CODES[inst.arm] in r._arms and rem[i] > 0]
    if not cands:
        unassigned.append(n); continue
    scored = sorted(((_sim(inst.text, ac.at[i, "claim_text"]), i) for i in cands), reverse=True)
    pick = None
    for s, i in scored:                          # pass 1: numeric guard on
        rn = _nums(ac.at[i, "claim_text"])
        if rn and not set(rn).issubset(set(_nums(inst.text))):
            continue
        pick = (s, i); break
    if pick is None:                             # pass 2: forced, flagged for provenance
        pick = scored[0]; forced.add(n)
    row_of[n], sim_of[n] = pick[1], pick[0]
    rem[pick[1]] -= 1

df["row_i"] = pd.Series(row_of)
df["sim"]   = pd.Series(sim_of)
df["manual_fix"] = False
print(f"assigned: {len(row_of)}/{len(df)} (forced: {len(forced)}, "
      f"low-sim<0.75: {sum(1 for s in sim_of.values() if s < 0.75)}, "
      f"unassigned: {len(unassigned)})")

# --- 3. manual overrides, adjudicated via JSON claim ids (AF_S6_Q2, AF_S6_Q6) ---
# The greedy matcher parked 4 HAS-BLED component / wording-variant claims in wrong rows
# and left 3 unassigned; audit CSV itself was verified correct against the JSON claim ids.
FIX = [
    ("AF_S6_Q2", 1, "coarse",            "Age over 65 years contributes", 14),
    ("AF_S6_Q2", 1, "atomic_gl_filter",  "Bleeding predisposition from anaemia", 14),
    ("AF_S6_Q2", 3, "atomic_gl_filter",  "A formal bleeding risk score is recommended", 12),
    ("AF_S6_Q2", 3, "atomic_sec_filter", "A formal bleeding risk score is recommended", 12),
    ("AF_S6_Q2", 3, "atomic_sec_filter", "The bleeding risk score should guide", 13),
    ("AF_S6_Q6", 3, "coarse",            "Oral anticoagulation is indicated", 27),
    ("AF_S6_Q6", 3, "atomic_sec_filter", "Preferred triple therapy is an oral anticoagulant", 28),
]
for uid, run, arm, pref, row in FIX:
    m = df[(df.uid == uid) & (df.run == run) & (df.arm == arm) & df.text.str.startswith(pref)]
    assert len(m) == 1, (uid, run, arm, pref, len(m))
    df.loc[m.index[0], ["row_i", "sim", "manual_fix"]] = [row, 1.0, True]
pd.DataFrame(FIX, columns=["uid", "run", "arm", "text_prefix", "row_i"]).to_csv(
    os.path.join(ANALYSIS_DIR, "b1_manual_overrides.csv"), index=False)
df["label"] = df["row_i"].map(ac["label"])
print(f"manual overrides applied: {len(FIX)}/7")

# --- 4. validation: per-row counts and label totals must equal the audit exactly ---
cnt = df.groupby("row_i").size()
bad = [i for i in ac.index if int(cnt.get(i, 0)) != int(ac.at[i, "_n_parsed"])]
EXPECT = {"true_hallucination": 284, "verifier_error": 235, "cross_source": 50,
          "version_mismatch": 37}
got = df.label.value_counts().to_dict()
print(f"rows where assigned != n_instances: {len(bad)} | unassigned: {df.label.isna().sum()}")
for k, v in EXPECT.items():
    g = got.get(k, 0)
    print(f"  {k:18s} got {g:4d}  expect {v:4d}  {'OK' if g == v else 'MISMATCH'}")
assert not bad and df.label.notna().all() and got == EXPECT, "assignment does not reconcile"

# provenance export of non-trivial assignments (forced / low-sim / manual)
chk_idx = sorted(n for n in df.index
                 if n in forced or (pd.notna(df.at[n, "sim"]) and df.at[n, "sim"] < 0.75)
                 or df.at[n, "manual_fix"])
out = df.loc[chk_idx, ["group", "run", "arm", "uid", "text", "row_i", "sim", "manual_fix"]].copy()
out["row_claim_id"] = out.row_i.map(ac.claim_id)
out["row_text"]     = out.row_i.map(ac.claim_text)
out.to_csv(os.path.join(ANALYSIS_DIR, "b1_check_assignment.csv"), index=False)

# --- 5. Step-1 tables ---
ARMS = list(ARM_CODES)
t1 = df.pivot_table(index="label", columns="arm", values="text",
                    aggfunc="count", fill_value=0).reindex(columns=ARMS, fill_value=0)
t1["TOTAL"] = t1.sum(axis=1)
t1.loc["TOTAL"] = t1.sum(axis=0)
print("\nT1: first-pass flagged instances, arm x adjudicated label")
display(t1)

print("per-arm totals and TH precision:")
for a in ARMS:
    tot = int((df.arm == a).sum())
    th  = int(((df.arm == a) & (df.label == "true_hallucination")).sum())
    print(f"  {a:18s} flagged {tot:4d}   TH {th:4d}   precision {th/tot:.1%}")

th_idx  = set(ac.index[ac.label == "true_hallucination"])
th_uids = set(ac.loc[ac.label == "true_hallucination", "uid"])
thd = df[df.row_i.isin(th_idx)]
print(f"\nT2: TH detection sensitivity per arm (rows: {len(th_idx)}, uids: {len(th_uids)})")
for a in ARMS:
    sub = thd[thd.arm == a]
    rows_caught = set(sub.row_i.dropna().astype(int))
    uid_any = set(ac.loc[list(rows_caught), "uid"]) if rows_caught else set()
    uid_all3 = {u for u in th_uids
                if all(((sub.uid == u) & (sub.run == r)).any() for r in (1, 2, 3))}
    print(f"  {a:18s} rows caught: {len(rows_caught):2d}/{len(th_idx)}   "
          f"uids >=1 run: {len(uid_any):2d}/{len(th_uids)}   "
          f"uids all 3 runs: {len(uid_all3):2d}/{len(th_uids)}")

df.to_csv(os.path.join(ANALYSIS_DIR, "step1_instances.csv"), index=False)
t1.to_csv(os.path.join(ANALYSIS_DIR, "step1_t1_arm_x_label.csv"))
print("\nsaved: step1_instances.csv, step1_t1_arm_x_label.csv, "
      "b1_manual_overrides.csv, b1_check_assignment.csv")

claim fields: ['chunk_id', 'claim', 'contradiction_kind', 'error_type', 'human_label', 'id', 'rationale', 'retrieved_ids', 'target_class', 'target_level', 'type', 'verdict'] | using: claim / verdict
first-pass refuted instances: 606 (expect 606) | all-iteration occurrences (diagnostic): 832
assigned: 603/606 (forced: 148, low-sim<0.75: 276, unassigned: 3)
manual overrides applied: 7/7
rows where assigned != n_instances: 0 | unassigned: 0
  true_hallucination got  284  expect  284  OK
  verifier_error     got  235  expect  235  OK
  cross_source       got   50  expect   50  OK
  version_mismatch   got   37  expect   37  OK

T1: first-pass flagged instances, arm x adjudicated label


arm,coarse,atomic,atomic_gl_filter,atomic_sec_filter,TOTAL
label,,,,,
cross_source,9,8,14,19,50
true_hallucination,63,72,71,78,284
verifier_error,58,67,62,48,235
version_mismatch,11,7,10,9,37
TOTAL,141,154,157,154,606


per-arm totals and TH precision:
  coarse             flagged  141   TH   63   precision 44.7%
  atomic             flagged  154   TH   72   precision 46.8%
  atomic_gl_filter   flagged  157   TH   71   precision 45.2%
  atomic_sec_filter  flagged  154   TH   78   precision 50.6%

T2: TH detection sensitivity per arm (rows: 59, uids: 39)
  coarse             rows caught: 42/59   uids >=1 run: 30/39   uids all 3 runs:  9/39
  atomic             rows caught: 42/59   uids >=1 run: 31/39   uids all 3 runs: 10/39
  atomic_gl_filter   rows caught: 43/59   uids >=1 run: 30/39   uids all 3 runs:  9/39
  atomic_sec_filter  rows caught: 47/59   uids >=1 run: 33/39   uids all 3 runs: 11/39

saved: step1_instances.csv, step1_t1_arm_x_label.csv, b1_manual_overrides.csv, b1_check_assignment.csv


In [ ]:
import os

path = "/content/drive/MyDrive/medical_qa_hallucination/experiments/audit"

for filename in os.listdir(path):
    print(filename)

th_confirmation_expert.xlsx
audit.csv
audit_claims.csv
th_review.xlsx
audit.ipynb
supported_probe


In [ ]:
# CELL B2 - Step 2: run-to-run detection stability
# Key is row_i (audit row index). Levels: claim row and uid, per arm.

ARMS = list(ARM_CODES)

def row_stability(label_filter, name):
    d = df if label_filter is None else df[df.label == label_filter]
    keys = list(ac.index if label_filter is None
                else ac.index[ac.label == label_filter])
    rows = []
    for a in ARMS:
        sub = d[d.arm == a]
        dist, n_total = {1: 0, 2: 0, 3: 0}, 0
        for i in keys:
            n_runs = sub[sub.row_i == i].run.nunique()
            if n_runs > 0:
                dist[n_runs] += 1
                n_total += 1
        rows.append(dict(arm=a, rows_flagged=n_total,
                         in_1_run=dist[1], in_2_runs=dist[2], in_3_runs=dist[3],
                         pct_all3=round(100 * dist[3] / max(n_total, 1), 1)))
    t = pd.DataFrame(rows).set_index("arm")
    print(f"\n{name}  (of {len(keys)} rows)")
    display(t)
    return t

t3 = row_stability(None, "T3: claim-row stability per arm")
t4 = row_stability("true_hallucination", "T4: TH claim-row stability per arm")

def uid_stability(label_filter, uids, name):
    d = df if label_filter is None else df[df.label == label_filter]
    rows = []
    for a in ARMS:
        sub = d[d.arm == a]
        dist, n_total = {1: 0, 2: 0, 3: 0}, 0
        for u in uids:
            n_runs = sub[sub.uid == u].run.nunique()
            if n_runs > 0:
                dist[n_runs] += 1
                n_total += 1
        rows.append(dict(arm=a, uids_flagged=n_total,
                         in_1_run=dist[1], in_2_runs=dist[2], in_3_runs=dist[3],
                         pct_all3=round(100 * dist[3] / max(n_total, 1), 1)))
    t = pd.DataFrame(rows).set_index("arm")
    print(f"\n{name}  (of {len(uids)} uids)")
    display(t)
    return t

all_flagged_uids = sorted(df.uid.unique())
th_uids = sorted(set(ac.loc[ac.label == "true_hallucination", "uid"]))
t5 = uid_stability(None, all_flagged_uids, "T5: uid stability per arm")
t6 = uid_stability("true_hallucination", th_uids, "T6: TH uid stability per arm")

with pd.ExcelWriter(os.path.join(ANALYSIS_DIR, "step2_stability.xlsx")) as xl:
    for name, t in [("T3_rows_all", t3), ("T4_rows_TH", t4),
                    ("T5_uids_all", t5), ("T6_uids_TH", t6)]:
        t.to_excel(xl, sheet_name=name)
print("\nsaved step2_stability.xlsx")


T3: claim-row stability per arm  (of 160 rows)


,rows_flagged,in_1_run,in_2_runs,in_3_runs,pct_all3
arm,,,,,
coarse,100,70,20,10,10.0
atomic,94,53,29,12,12.8
atomic_gl_filter,104,67,26,11,10.6
atomic_sec_filter,102,67,25,10,9.8



T4: TH claim-row stability per arm  (of 59 rows)


,rows_flagged,in_1_run,in_2_runs,in_3_runs,pct_all3
arm,,,,,
coarse,42,26,11,5,11.9
atomic,42,22,15,5,11.9
atomic_gl_filter,43,24,14,5,11.6
atomic_sec_filter,47,27,14,6,12.8



T5: uid stability per arm  (of 91 uids)


,uids_flagged,in_1_run,in_2_runs,in_3_runs,pct_all3
arm,,,,,
coarse,65,34,13,18,27.7
atomic,65,28,15,22,33.8
atomic_gl_filter,66,29,18,19,28.8
atomic_sec_filter,65,28,16,21,32.3



T6: TH uid stability per arm  (of 39 uids)


,uids_flagged,in_1_run,in_2_runs,in_3_runs,pct_all3
arm,,,,,
coarse,30,14,7,9,30.0
atomic,31,12,9,10,32.3
atomic_gl_filter,30,13,8,9,30.0
atomic_sec_filter,33,13,9,11,33.3



saved step2_stability.xlsx


In [ ]:
# CELL B3 - Step 3: correction-loop outcomes
# (a) mechanical loop stats from run payloads (ref_init / final_refuted / iterations)
# (b) adjudicated outcomes: outcome narratives in audit_claims.csv are mapped to a
#     structured outcome_class by keyword rules; 5 rows where no keyword fired were
#     adjudicated manually (documented in OVERRIDE_OC) - labels themselves unchanged

# --- (a) mechanical: convergence and loop usage ---
rows = []
for tag, pl in RUNS.items():
    group, run = tag.rsplit("_run", 1)
    for q in pl["results"]:
        for arm, ares in q["arms"].items():
            rows.append(dict(group=group, run=int(run), arm=arm, uid=q["uid"],
                             ref_init=ares["ref_init"],
                             final_refuted=ares["final_refuted"],
                             iterations=ares["iterations"]))
lp = pd.DataFrame(rows)
print(f"question-arm-run instances: {len(lp)} (expect 1920)")

print("\nloop stats per arm (instances with ref_init > 0):")
for a in ARMS:
    sub = lp[lp.arm == a]
    flagged = sub[sub.ref_init > 0]
    conv = flagged[flagged.final_refuted == 0]
    print(f"  {a:18s} flagged {len(flagged):4d}   converged {len(conv):4d} "
          f"({len(conv)/len(flagged):.1%})   unresolved {len(flagged)-len(conv):3d}   "
          f"mean iters {flagged.iterations.mean():.2f}")
    print(f"      iterations distribution: {flagged.iterations.value_counts().sort_index().to_dict()}")
lp.to_csv(os.path.join(ANALYSIS_DIR, "step3_loop_stats.csv"), index=False)

# --- (b) structured outcome_class from narrative outcome column ---
def classify_outcome(o):
    t = str(o).lower()
    if ("correction-introduced" in t or "correction loop manufactured" in t
            or "correction failure cascade" in t or "correction-failure ship" in t
            or "converged but wrong" in t or "converged but worse" in t
            or ("capitulat" in t and "contradict" in t)):
        return "correction_harm"
    if "unresolved" in t or "failed to converge" in t:
        return "unresolved"
    if t.startswith(("mixed", "split", "partial")):
        return "mixed"
    if ("survived" in t or "resisted" in t or "no damage" in t
            or "correctly kept" in t or "kept correct" in t):
        return "resisted_kept"
    if ("dropped" in t or "lost" in t or "narrowed" in t or "deletion" in t
            or "went qualitative" in t or "hedged" in t or "softer" in t):
        return "narrowed"
    if t.startswith("converged") or "converged;" in t:
        return "corrected_clean"
    return "other"

ac["outcome_class"] = ac.outcome.map(classify_outcome)

# manual adjudication of the 5 rows where no keyword fired (agreed in analysis chat):
#   6  VHD_S9_Q1 A  ve   finals narrowed to generic framing, no contradiction -> narrowed
#   8  VHD_S12_Q1 A ve   claim kept in all 8 finals, 2 collateral inversions  -> mixed
#   27 AF_S6_Q6  A  ve   claim substance preserved; uncaught error adjacent   -> resisted_kept
#   83 CVD_S6_Q1 B  cs   ag converged correctly, as landed corpus-inconsistent -> mixed
#   101 AF_S3_Q3 A  th   2 of 3 runs corrected, run1 shipped refuted 175       -> mixed
OVERRIDE_OC = {6: "narrowed", 8: "mixed", 27: "resisted_kept", 83: "mixed", 101: "mixed"}
for i, oc in OVERRIDE_OC.items():
    assert ac.at[i, "outcome_class"] == "other"
    ac.at[i, "outcome_class"] = oc
assert (ac.outcome_class != "other").all(), "unclassified outcome rows remain"

print("\noutcome_class x label (160 rows):")
t7 = pd.crosstab(ac.outcome_class, ac.label, margins=True)
display(t7)

print("\ncorrection-loop fate of the 59 TH rows:")
for k, v in ac.loc[ac.label == "true_hallucination", "outcome_class"].value_counts().items():
    print(f"  {k:16s} {v:3d}  ({v/59:.1%})")

ac[["uid", "claim_id", "outcome_class"]].to_csv(
    os.path.join(ANALYSIS_DIR, "step3_outcome_class_final.csv"), index=False)
t7.to_csv(os.path.join(ANALYSIS_DIR, "step3_outcome_class_x_label.csv"))
print("\nsaved: step3_loop_stats.csv, step3_outcome_class_final.csv, step3_outcome_class_x_label.csv")
print("manual follow-up: merge outcome_class into audit_claims.csv (audit-phase column)")

question-arm-run instances: 1920 (expect 1920)

loop stats per arm (instances with ref_init > 0):
  coarse             flagged  114   converged   98 (86.0%)   unresolved  16   mean iters 2.25
      iterations distribution: {2: 86, 3: 28}
  atomic             flagged  124   converged  111 (89.5%)   unresolved  13   mean iters 2.23
      iterations distribution: {2: 95, 3: 29}
  atomic_gl_filter   flagged  122   converged  104 (85.2%)   unresolved  18   mean iters 2.30
      iterations distribution: {2: 86, 3: 36}
  atomic_sec_filter  flagged  123   converged  117 (95.1%)   unresolved   6   mean iters 2.20
      iterations distribution: {2: 99, 3: 24}

outcome_class x label (160 rows):


label,cross_source,true_hallucination,verifier_error,version_mismatch,All
outcome_class,,,,,
corrected_clean,3,38,18,6,65
correction_harm,0,4,15,0,19
mixed,3,2,5,0,10
narrowed,11,8,17,0,36
resisted_kept,1,1,12,1,15
unresolved,0,6,8,1,15
All,18,59,75,8,160



correction-loop fate of the 59 TH rows:
  corrected_clean   38  (64.4%)
  narrowed           8  (13.6%)
  unresolved         6  (10.2%)
  correction_harm    4  (6.8%)
  mixed              2  (3.4%)
  resisted_kept      1  (1.7%)

saved: step3_loop_stats.csv, step3_outcome_class_final.csv, step3_outcome_class_x_label.csv
manual follow-up: merge outcome_class into audit_claims.csv (audit-phase column)


In [ ]:
# CELL B4 - Step 4: cross-arm detection consistency
# Per audit row: which arms flagged it (>=1 instance). Uses df, ac, ARMS.

import itertools

ARMS = list(ARM_CODES)
ac["_armset"] = pd.Series(df.groupby("row_i")["arm"].agg(set))
ac["_n_arms"] = ac._armset.map(lambda s: len(s) if isinstance(s, set) else 0)
assert ac._n_arms.gt(0).all()   # every audit row was flagged by at least one arm

# --- T8: consensus level (in how many arms flagged) x label ---
t8 = pd.crosstab(ac._n_arms, ac.label, margins=True)
t8["TH_share"] = (t8["true_hallucination"] / t8["All"]).round(3)
print("T8: rows by number of arms flagging them, x label")
display(t8)

# --- T9: TH rows (59) - per-arm sets, union, unique contributions, pairwise overlap ---
th_rows = set(ac.index[ac.label == "true_hallucination"])
sets = {a: set(df.loc[df.arm.eq(a) & df.row_i.isin(th_rows), "row_i"].astype(int))
        for a in ARMS}
union = set().union(*sets.values())
inter = set.intersection(*sets.values())
print(f"\nT9: TH rows caught (of {len(th_rows)})")
for a in ARMS:
    others = set().union(*(sets[b] for b in ARMS if b != a))
    print(f"  {a:18s} caught {len(sets[a]):2d}   unique to this arm: {len(sets[a] - others)}")
print(f"  union (>=1 arm)      {len(union):2d}   "
      f"intersection (all 4)  {len(inter):2d}")
print("  pairwise Jaccard overlap of caught-row sets:")
for a, b in itertools.combinations(ARMS, 2):
    j = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
    print(f"    {a:18s} vs {b:18s} {j:.2f}")

# --- T10: TH uids (39) - union across arms ---
th_uids = set(ac.loc[ac.label == "true_hallucination", "uid"])
uid_sets = {a: set(ac.loc[list(sets[a]), "uid"]) for a in ARMS}
union_u = set().union(*uid_sets.values())
print(f"\nT10: TH uids caught (of {len(th_uids)})")
for a in ARMS:
    print(f"  {a:18s} {len(uid_sets[a]):2d}")
print(f"  union (>=1 arm) {len(union_u):2d}/{len(th_uids)}")

# --- consensus as a detection strategy: rows flagged by >=k arms ---
print("\nconsensus strategy on TH rows (flagged by >=k of 4 arms):")
for k in (1, 2, 3, 4):
    sub = ac[ac._n_arms >= k]
    n_th = (sub.label == "true_hallucination").sum()
    print(f"  >={k} arms: {len(sub):3d} rows flagged, {n_th:3d} TH "
          f"(precision {n_th/len(sub):.1%}, TH-row recall {n_th/len(th_rows):.1%})")

t8.to_csv(os.path.join(ANALYSIS_DIR, "step4_consensus_x_label.csv"))
print("\nsaved step4_consensus_x_label.csv")

T8: rows by number of arms flagging them, x label


label,cross_source,true_hallucination,verifier_error,version_mismatch,All,TH_share
_n_arms,,,,,,
1,6,13,28,3,50,0.260
2,5,8,21,1,35,0.229
3,4,7,8,1,20,0.350
4,3,31,18,3,55,0.564
All,18,59,75,8,160,0.369



T9: TH rows caught (of 59)
  coarse             caught 42   unique to this arm: 5
  atomic             caught 42   unique to this arm: 2
  atomic_gl_filter   caught 43   unique to this arm: 0
  atomic_sec_filter  caught 47   unique to this arm: 6
  union (>=1 arm)      59   intersection (all 4)  31
  pairwise Jaccard overlap of caught-row sets:
    coarse             vs atomic             0.65
    coarse             vs atomic_gl_filter   0.70
    coarse             vs atomic_sec_filter  0.59
    atomic             vs atomic_gl_filter   0.81
    atomic             vs atomic_sec_filter  0.71
    atomic_gl_filter   vs atomic_sec_filter  0.76

T10: TH uids caught (of 39)
  coarse             30
  atomic             31
  atomic_gl_filter   30
  atomic_sec_filter  33
  union (>=1 arm) 39/39

consensus strategy on TH rows (flagged by >=k of 4 arms):
  >=1 arms: 160 rows flagged,  59 TH (precision 36.9%, TH-row recall 100.0%)
  >=2 arms: 110 rows flagged,  46 TH (precision 41.8%, TH-row recal

In [ ]:
# CELL B5 - Step 5: residual errors in shipped final answers
# (a) finals shipped with unresolved refuted claims; (b) their adjudicated labels;
# (c) correction-introduced harm rows from Step 3

# --- (a) finals with final_refuted > 0 ---
unres = lp[lp.final_refuted > 0]
print(f"finals with unresolved refuted claims: {len(unres)} / 1920")
print(unres.arm.value_counts().reindex(ARMS).to_string())

# --- (b) extract final-iteration refuted claims, match to audit rows ---
fin = []
for tag, pl in RUNS.items():
    group, run = tag.rsplit("_run", 1)
    for q in pl["results"]:
        for arm, ares in q["arms"].items():
            if ares["final_refuted"] > 0:
                for c in ares["history"][-1]["verified"]:
                    if str(c[K_V]).strip().lower() == "refuted":
                        fin.append(dict(group=group, run=int(run), arm=arm,
                                        uid=q["uid"], text=str(c[K_TX])))
fd = pd.DataFrame(fin)
print(f"\nfinal-iteration refuted claims: {len(fd)} "
      f"(sum of final_refuted: {int(lp.final_refuted.sum())})")

def _match_row(uid, text):      # best-similarity row within uid, numeric guard, no capacity
    best, bs = None, 0.0
    for i in ac.index[ac.uid == uid]:
        rn = _nums(ac.at[i, "claim_text"])
        if rn and not set(rn).issubset(set(_nums(text))):
            continue
        s = _sim(text, ac.at[i, "claim_text"])
        if s > bs:
            best, bs = i, s
    return best, bs

m = fd.apply(lambda r: _match_row(r.uid, r.text), axis=1)
fd["row_i"] = [x[0] for x in m]
fd["sim"]   = [x[1] for x in m]
fd["label"] = fd.row_i.map(ac.label)
fd.loc[fd.sim < 0.60, ["row_i", "label"]] = None   # weak match: treat as new claim

print("\nshipped refuted claims by adjudicated label:")
print(fd.label.value_counts(dropna=False).to_string())
print("\nper arm x label:")
display(fd.pivot_table(index="label", columns="arm", values="text",
                       aggfunc="count", fill_value=0, margins=True,
                       margins_name="TOTAL").reindex(columns=ARMS + ["TOTAL"]))

new_claims = fd[fd.row_i.isna()]
print(f"\nshipped refuted claims NOT matching any audit row: {len(new_claims)} "
      f"(candidates for correction-introduced errors)")
for _, r in new_claims.iterrows():
    print(f"  {r.uid:14s} run{r.run} {r.arm:18s} {r.text[:130]}")

# --- (c) correction-introduced harm (from Step 3 adjudicated outcomes) ---
harm = ac[ac.outcome_class == "correction_harm"]
print(f"\ncorrection_harm rows (answer-level damage events): {len(harm)} "
      f"(verifier_error-driven: {(harm.label=='verifier_error').sum()}, "
      f"TH-driven: {(harm.label=='true_hallucination').sum()})")

fd.to_csv(os.path.join(ANALYSIS_DIR, "step5_shipped_refuted.csv"), index=False)
print("\nsaved step5_shipped_refuted.csv")

finals with unresolved refuted claims: 53 / 1920
arm
coarse               16
atomic               13
atomic_gl_filter     18
atomic_sec_filter     6

final-iteration refuted claims: 76 (sum of final_refuted: 76)

shipped refuted claims by adjudicated label:
label
None                  56
true_hallucination    11
verifier_error         9

per arm x label:


arm,coarse,atomic,atomic_gl_filter,atomic_sec_filter,TOTAL
label,,,,,
true_hallucination,3,3,4,1,11
verifier_error,5,1,3,0,9
TOTAL,8,4,7,1,20



shipped refuted claims NOT matching any audit row: 56 (candidates for correction-introduced errors)
  AF_S3_Q1       run1 atomic             The key criterion for classifying atrial fibrillation as paroxysmal is spontaneous self-termination within 7 days.
  AF_S3_Q1       run1 atomic             EHRA IIa describes mild symptoms that are present but not troublesome.
  AF_S6_Q2       run1 atomic_sec_filter  A formal bleeding-risk score should be used to support structured assessment and follow-up.
  AF_S6_Q2       run1 atomic_sec_filter  HAS-BLED is the guideline-preferred tool for bleeding-risk assessment.
  VHD_S8_Q9      run2 atomic_gl_filter   Very severe aortic stenosis (AS) is defined by a mean transvalvular gradient of 60 mmHg or higher according to ESC criteria.
  AF_S3_Q1       run3 coarse             mEHRA class 2 corresponds to patients with noticeable symptoms but preserved normal daily activity.
  AF_S9_Q11      run1 atomic_gl_filter   Both peri-operative amiodarone and pos

In [ ]:
# CELL B5r - export review table for the unclear shipped-refuted claims
import json
from pathlib import Path

# --- load chunk texts from the frozen RAG index (same files as audit A4) ---
RAG_DIR = Path(config.RESULTS_DIR).parent / "rag_index"
chunk_text = {}
for p in sorted(RAG_DIR.glob("*.jsonl")):
    with open(p) as f:
        for line in f:
            r = json.loads(line)
            cid = r.get("chunk_id") or r.get("id")
            txt = r.get("text") or r.get("content") or ""
            if cid is not None:
                chunk_text[str(cid)] = str(txt)
print("chunks loaded:", len(chunk_text), "from", [p.name for p in sorted(RAG_DIR.glob('*.jsonl'))])

def _excerpt(cid, n=400):
    cid = str(cid or "")
    for k in (cid, cid.split("->")[0],
              cid.replace("atomic: ", "atomic:").replace("coarse: ", "coarse:")):
        if k in chunk_text:
            return chunk_text[k][:n]
    return ""

# --- re-extract final-iteration refuted claims with full verifier context ---
fin_full = []
for tag, pl in RUNS.items():
    group, run = tag.rsplit("_run", 1)
    for q in pl["results"]:
        for arm, ares in q["arms"].items():
            if ares["final_refuted"] > 0:
                for c in ares["history"][-1]["verified"]:
                    if str(c[K_V]).strip().lower() == "refuted":
                        fin_full.append(dict(group=group, run=int(run), arm=arm,
                                             uid=q["uid"], claim=str(c[K_TX]),
                                             rationale=str(c.get("rationale", "")),
                                             contradiction_kind=str(c.get("contradiction_kind", "")),
                                             chunk_id=str(c.get("chunk_id", "")),
                                             retrieved=str(c.get("retrieved_ids", ""))))
assert len(fin_full) == len(fd)
ffd = pd.DataFrame(fin_full)
ffd["chunk_excerpt"] = ffd.chunk_id.map(_excerpt)
ffd["audit_rows_of_uid"] = ffd.uid.map(
    lambda u: " | ".join(f"{r.claim_id}[{r.label}]: {str(r.claim_text)[:80]}"
                         for _, r in ac[ac.uid == u].iterrows()))

# --- the 25 needing review (confirmed ones are excluded) ---
REVIEW = [
    ("AF_S9_Q11", 1, "atomic_gl_filter", "Both peri-operative amiodarone"),
    ("AF_S9_Q11", 2, "coarse",           "For peri-operative prevention"),
    ("AF_S9_Q11", 2, "coarse",           "The recommendations for peri-operative"),
    ("AF_S9_Q11", 2, "atomic",           "For peri-operative prevention"),
    ("AF_S9_Q11", 2, "atomic",           "Colchicine may be considered"),
    ("AF_S9_Q11", 2, "atomic",           "Beta-blockers have a positive"),
    ("AF_S9_Q11", 2, "atomic_sec_filter","Colchicine may be considered"),
    ("AF_S9_Q11", 3, "coarse",           "Colchicine may be considered"),
    ("AF_S9_Q11", 3, "coarse",           "The preventive strategies"),
    ("AF_S9_Q5",  2, "atomic_gl_filter", "In an AF patient undergoing PCI"),
    ("AF_S9_Q5",  2, "atomic_gl_filter", "A reduced DOAC dose"),
    ("VHD_S9_Q2", 1, "coarse",           "An effective regurgitant orifice area (EROA) of 80"),
    ("VHD_S9_Q2", 2, "atomic",           "Severe secondary mitral regurgitation"),
    ("VHD_S9_Q2", 3, "atomic_gl_filter", "Severe secondary mitral regurgitation"),
    ("VHD_S14_Q2",1, "atomic",           "In the ESC framework, severe haemodynamic"),
    ("VHD_S14_Q2",2, "atomic",           "In the ESC framework, severe mitral BHV"),
    ("VHD_S14_Q2",3, "coarse",           "Severe haemodynamic SVD is defined"),
    ("CVD_S4-6_Q2",1,"coarse",           "Adding ezetimibe"),
    ("CVD_S4-6_Q2",1,"atomic_gl_filter", "The main expected LDL-C reductions"),
    ("CVD_S4-6_Q2",1,"atomic_sec_filter","The main expected LDL-C reductions"),
    ("AF_S9_Q8",  2, "atomic_gl_filter", "The P2Y12 inhibitor"),
    ("VHD_S8_Q8", 2, "atomic_sec_filter","Outcomes with intervention"),
    ("CM_S8_Q1",  2, "atomic_gl_filter", "In individuals with dilated or non-dilated"),
    ("CVD_S3-3_Q1",1,"atomic_gl_filter", "For Black ethnic groups"),
    ("CVD_S4-7_Q1",3,"coarse",           "For adults aged 65"),
]
rev_rows = []
for uid, run, arm, pref in REVIEW:
    m = ffd[(ffd.uid == uid) & (ffd.run == run) & (ffd.arm == arm)
            & ffd.claim.str.startswith(pref)]
    assert len(m) == 1, (uid, run, arm, pref, len(m))
    rev_rows.append(m.iloc[0])
rev = pd.DataFrame(rev_rows)[["uid", "run", "arm", "claim", "rationale",
                              "contradiction_kind", "chunk_id", "chunk_excerpt",
                              "audit_rows_of_uid"]]
rev.to_csv(os.path.join(ANALYSIS_DIR, "step5_review_unclear.csv"), index=False)
print(f"saved step5_review_unclear.csv ({len(rev)} rows)")
miss = rev[rev.chunk_excerpt == ""]
print(f"rows without chunk excerpt: {len(miss)}")
if len(miss):
    print(miss[["uid", "chunk_id"]].to_string())

chunks loaded: 0 from ['atomic_meta.jsonl', 'coarse_meta.jsonl']
saved step5_review_unclear.csv (25 rows)
rows without chunk excerpt: 25
            uid                                                        chunk_id
14    AF_S9_Q11           atomic:0471:AF:recommendation:Recommendation_Table_26
18    AF_S9_Q11                                                            0265
19    AF_S9_Q11              coarse:0265:AF:table_whole:Recommendation_Table_26
21    AF_S9_Q11           atomic:0474:AF:recommendation:Recommendation_Table_26
22    AF_S9_Q11                                                            0452
23    AF_S9_Q11           atomic:0474:AF:recommendation:Recommendation_Table_26
24    AF_S9_Q11     atomic:0452:AF:prose:9.6 AF-CARE in post-operative patients
25    AF_S9_Q11     coarse:0254:AF:prose:9.6 AF-CARE in post-operative patients
26    AF_S9_Q11     coarse:0254:AF:prose:9.6 AF-CARE in post-operative patients
16     AF_S9_Q5           atomic:0445:AF:prose:9.2 AF-CARE in a

In [ ]:
# CELL B5b - lock shipped-claim adjudication, bottom-line residual-error table
# shipped_class: genuine (real error shipped) / phantom (refuted but content fine)
# origin: unresolved_TH (audited TH persisted), CIE (correction-introduced),
#         version_mismatch, new (new claim, not audited, not provably loop-made)
CLS = [
    # ---- 23 REST genuine ----
    ("AF_S3_Q1",1,"atomic","The key criterion for classifying","genuine","version_mismatch"),
    ("AF_S6_Q2",1,"atomic_sec_filter","A formal bleeding-risk score","genuine","version_mismatch"),
    ("AF_S6_Q2",1,"atomic_sec_filter","HAS-BLED is the guideline-preferred","genuine","version_mismatch"),
    ("AF_S10_Q3",1,"coarse","Approximately 37.6 million","genuine","CIE"),
    ("AF_S10_Q3",1,"coarse","In a United States community cohort","genuine","CIE"),
    ("VHD_S6_Q4",2,"coarse","Concomitant surgical AF ablation","genuine","unresolved_TH"),
    ("VHD_S6_Q7",2,"atomic_gl_filter","cardiogenic shock is present","genuine","unresolved_TH"),
    ("VHD_S6_Q7",3,"atomic_gl_filter","cardiogenic shock complicates","genuine","unresolved_TH"),
    ("CM_S7-1_Q3",2,"atomic_sec_filter","The presence of a left ventricular apical","genuine","unresolved_TH"),
    ("AF_S7_Q11",3,"atomic","Short-term post-cardioversion","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",1,"coarse","Patients with dilated and nondilated","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",2,"coarse","In arrhythmogenic right ventricular","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",2,"coarse","In dilated cardiomyopathy","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",2,"coarse","In non-dilated left ventricular","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",2,"coarse","The key distinction in anticoagulation","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",3,"coarse","In arrhythmogenic right ventricular","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",3,"coarse","In dilated or non-dilated","genuine","CIE"),
    ("CM_S6-10-6-11_Q1",3,"coarse","The cardiomyopathy phenotypes managed","genuine","CIE"),
    ("CVD_S4-7_Q1",2,"coarse","If younger patients have diabetes","genuine","unresolved_TH"),
    ("VHD_S9_Q5",2,"atomic","If the patient is symptomatic, mitral valve surgery","genuine","unresolved_TH"),
    ("CVD_S3-3_Q1",2,"atomic","For Bangladeshi ethnicity","genuine","CIE"),
    ("CVD_S3-3_Q1",2,"atomic","For Pakistani ethnicity","genuine","CIE"),
    ("CVD_S3-3_Q1",3,"atomic_gl_filter","For people of Bangladeshi origin","genuine","CIE"),
    # ---- 8 REST phantom ----
    ("AF_S3_Q1",1,"atomic","EHRA IIa describes mild symptoms","phantom",""),
    ("VHD_S8_Q9",2,"atomic_gl_filter","Very severe aortic stenosis (AS) is defined","phantom",""),
    ("AF_S3_Q1",3,"coarse","mEHRA class 2 corresponds","phantom",""),
    ("VHD_S9_Q2",1,"atomic_gl_filter","The classification of 'severe' mitral","phantom",""),
    ("AF_S9_Q8",1,"atomic","After ACS/PCI, the combination","phantom",""),
    ("AF_S5_Q1",2,"atomic","Dapagliflozin or empagliflozin","phantom",""),
    ("VHD_S8_Q4",2,"atomic","Severe aortic stenosis is highly likely","phantom",""),
    ("VHD_S8_Q4",3,"atomic_sec_filter","Severe AS is supported by an Agatston","phantom",""),
    # ---- 15 REVIEW genuine ----
    ("AF_S9_Q11",1,"atomic_gl_filter","Both peri-operative amiodarone","genuine","unresolved_TH"),
    ("AF_S9_Q11",2,"coarse","For peri-operative prevention","genuine","unresolved_TH"),
    ("AF_S9_Q11",2,"coarse","The recommendations for peri-operative","genuine","unresolved_TH"),
    ("AF_S9_Q11",2,"atomic","For peri-operative prevention","genuine","unresolved_TH"),
    ("AF_S9_Q11",2,"atomic","Colchicine may be considered","genuine","new"),
    ("AF_S9_Q11",2,"atomic","Beta-blockers have a positive","genuine","unresolved_TH"),
    ("AF_S9_Q11",2,"atomic_sec_filter","Colchicine may be considered","genuine","new"),
    ("AF_S9_Q11",3,"coarse","Colchicine may be considered","genuine","new"),
    ("AF_S9_Q11",3,"coarse","The preventive strategies","genuine","unresolved_TH"),
    ("CVD_S4-6_Q2",1,"atomic_gl_filter","The main expected LDL-C reductions","genuine","CIE"),
    ("CVD_S4-6_Q2",1,"atomic_sec_filter","The main expected LDL-C reductions","genuine","CIE"),
    ("VHD_S8_Q8",2,"atomic_sec_filter","Outcomes with intervention","genuine","unresolved_TH"),
    ("CM_S8_Q1",2,"atomic_gl_filter","In individuals with dilated or non-dilated","genuine","CIE"),
    ("CVD_S3-3_Q1",1,"atomic_gl_filter","For Black ethnic groups","genuine","unresolved_TH"),
    ("CVD_S4-7_Q1",3,"coarse","For adults aged 65","genuine","unresolved_TH"),
    # ---- 10 REVIEW phantom ----
    ("AF_S9_Q5",2,"atomic_gl_filter","In an AF patient undergoing PCI","phantom",""),
    ("AF_S9_Q5",2,"atomic_gl_filter","A reduced DOAC dose","phantom",""),
    ("VHD_S9_Q2",1,"coarse","An effective regurgitant orifice area","phantom",""),
    ("VHD_S9_Q2",2,"atomic","Severe secondary mitral regurgitation","phantom",""),
    ("VHD_S9_Q2",3,"atomic_gl_filter","Severe secondary mitral regurgitation","phantom",""),
    ("VHD_S14_Q2",1,"atomic","In the ESC framework, severe haemodynamic","phantom",""),
    ("VHD_S14_Q2",2,"atomic","In the ESC framework, severe mitral BHV","phantom",""),
    ("VHD_S14_Q2",3,"coarse","Severe haemodynamic SVD is defined","phantom",""),
    ("CVD_S4-6_Q2",1,"coarse","Adding ezetimibe","phantom",""),
    ("AF_S9_Q8",2,"atomic_gl_filter","The P2Y12 inhibitor","phantom",""),
]

fd["shipped_class"] = None
fd["origin"] = ""
for uid, run, arm, pref, cls, orig in CLS:
    m = fd[(fd.uid == uid) & (fd.run == run) & (fd.arm == arm)
           & fd.text.str.contains(pref, regex=False)]
    assert len(m) == 1 and fd.loc[m.index[0], "shipped_class"] is None, (uid, run, arm, pref)
    fd.loc[m.index[0], ["shipped_class", "origin"]] = [cls, orig]

# matched rows: audit label decides (TH -> genuine unresolved_TH; verifier_error -> phantom)
mm = fd.shipped_class.isna() & fd.row_i.notna()
fd.loc[mm & (fd.label == "true_hallucination"), ["shipped_class", "origin"]] = ["genuine", "unresolved_TH"]
fd.loc[mm & (fd.label == "verifier_error"), "shipped_class"] = "phantom"
assert fd.shipped_class.notna().all() and (fd.shipped_class != "").all()
print(f"76 shipped refuted claims classified: "
      f"genuine {(fd.shipped_class=='genuine').sum()}, phantom {(fd.shipped_class=='phantom').sum()}")

# --- bottom line per arm ---
print("\nfinals shipping >=1 genuine residual error (per 480 finals per arm):")
for a in ARMS:
    sub = fd[fd.arm == a]
    g = sub[sub.shipped_class == "genuine"]
    fin_g = g[["uid", "run"]].drop_duplicates()
    fin_p = sub[sub.shipped_class == "phantom"][["uid", "run"]].drop_duplicates()
    only_phantom = fin_p.merge(fin_g, on=["uid", "run"], how="left", indicator=True)
    n_only = (only_phantom._merge == "left_only").sum()
    print(f"  {a:18s} genuine-error finals {len(fin_g):3d}/480   "
          f"genuine claims {len(g):3d}   phantom-only finals {n_only:3d}")

print("\ngenuine claims by origin:")
print(fd[fd.shipped_class == "genuine"].origin.value_counts().to_string())

fd.to_csv(os.path.join(ANALYSIS_DIR, "step5_shipped_refuted.csv"), index=False)
print("\nsaved step5_shipped_refuted.csv (with shipped_class + origin)")

76 shipped refuted claims classified: genuine 49, phantom 27

finals shipping >=1 genuine residual error (per 480 finals per arm):
  coarse             genuine-error finals  10/480   genuine claims  20   phantom-only finals   6
  atomic             genuine-error finals   6/480   genuine claims  11   phantom-only finals   7
  atomic_gl_filter   genuine-error finals  10/480   genuine claims  11   phantom-only finals   8
  atomic_sec_filter  genuine-error finals   6/480   genuine claims   7   phantom-only finals   0

genuine claims by origin:
origin
unresolved_TH       26
CIE                 17
version_mismatch     3
new                  3

saved step5_shipped_refuted.csv (with shipped_class + origin)


In [ ]:
# ============================================================
# CELL B6 - Step 6: cost accounting (tokens, loop effort, spend)
#   phases kept separate:
#     - core experiment (OpenAI): dashboard $ + saved payload tokens
#     - dataset construction (Qwen3Max): bank $ (see qwen_usage.ipynb)
#   writes: step6_tokens.csv, step6_loop_effort.csv, step6_cost.csv
# ============================================================
import re

def _run_paths():
    # flatten RUN_FILES whatever its key shape; parse group/run from filename
    paths = []
    for v in RUN_FILES.values():
        paths.extend(v if isinstance(v, (list, tuple)) else [v])
    paths = sorted(set(paths))
    out = []
    for p in paths:
        m = re.match(r"results_(.+?)_run(\d+)", Path(p).name)
        assert m, f"unexpected filename: {p}"
        out.append((m.group(1), int(m.group(2)), Path(p)))
    return sorted(out)

def _arm_items(q):
    # arms is a dict {arm_name: result} in these payloads; tolerate a list too
    arms = q["arms"]
    return arms.items() if isinstance(arms, dict) else [(a["arm"], a) for a in arms]

RUNS_FLAT = _run_paths()
print(f"{len(RUNS_FLAT)} run files")   # expect 15

# ---------- Part A: tokens from saved run payloads ----------
rows = []
for group, run, path in RUNS_FLAT:
    p = json.loads(path.read_text())
    t = p.get("tokens_this_run") or {}
    rows.append(dict(group=group, run=run,
                     n_questions=p.get("n_questions"),
                     n_failed=p.get("n_failed"),
                     tok_in=t.get("in"), tok_out=t.get("out")))
tok = pd.DataFrame(rows)
print(tok.to_string(index=False))

tot = tok[["tok_in", "tok_out"]].sum()
b   = tok[tok.group != "reviewed20"][["tok_in", "tok_out"]].sum()
r   = tok[tok.group == "reviewed20"][["tok_in", "tok_out"]].sum()
print(f"\nall 15 runs: {tot.tok_in:,} in / {tot.tok_out:,} out")
print(f"batches 1-4: {b.tok_in:,} in / {b.tok_out:,} out")
print(f"reviewed20:  {r.tok_in:,} in / {r.tok_out:,} out")
tok.to_csv(ANALYSIS_DIR / "step6_tokens.csv", index=False)

# ---------- Part B: loop effort (all 15 runs) ----------
loop_rows = []
for group, run, path in RUNS_FLAT:
    p = json.loads(path.read_text())
    for q in p["results"]:
        for arm_name, a in _arm_items(q):
            loop_rows.append(dict(
                group=group, run=run, uid=q["uid"], arm=arm_name,
                iterations=a["iterations"],
                rewrite=a["iterations"] > 1,
                n_final_refuted=(a.get("final_refuted") if isinstance(a.get("final_refuted"), int)
                                 else len(a.get("final_refuted") or [])),
            ))
le = pd.DataFrame(loop_rows)
le["final_clean"] = le.n_final_refuted == 0
t_eff = (le.groupby("arm")
           .agg(n=("uid", "size"),
                mean_iter=("iterations", "mean"),
                pct_rewrite=("rewrite", lambda s: 100 * s.mean()),
                pct_final_clean=("final_clean", lambda s: 100 * s.mean()))
           .round(2))
print("\nloop effort per arm:"); print(t_eff)
print(f"overall mean iterations: {le.iterations.mean():.2f}")
le.to_csv(ANALYSIS_DIR / "step6_loop_effort.csv", index=False)

# ---------- Part C: initial claim volume ----------
icl = pd.Series([len(q.get("init_claims") or [])
                 for group, run, path in RUNS_FLAT
                 for q in json.loads(path.read_text())["results"]])
print(f"\ninit claims per question-instance: mean {icl.mean():.1f}, "
      f"median {icl.median():.0f}, max {icl.max()}")

# ---------- Part D: cost accounting ----------
# D1. core experiment (OpenAI) - dashboard records, authoritative $
openai_spend = pd.DataFrame([
    dict(date="2026-06-17", usd=1.75,  dash_in=2_960_000,  dash_out=150_000,
         scope="dev + pilot"),
    dict(date="2026-06-29", usd=5.91,  dash_in=12_600_000, dash_out=423_000,
         scope="reviewed20 round (3 runs x 4 arms) + dev same day"),
    dict(date="2026-09-18", usd=27.76, dash_in=53_700_000, dash_out=2_344_000,
         scope="batches 1-4 (140q x 3 runs x 4 arms = 1680 arm-runs)"),
])

# D2. consistency check: saved payload tokens vs dashboard (2026-09-18)
dash_in, dash_out = 53_700_000, 2_344_000
print(f"\nconsistency check (2026-09-18):")
print(f"  payload (batches): {b.tok_in:,} in / {b.tok_out:,} out")
print(f"  dashboard:         {dash_in:,} in / {dash_out:,} out")
print(f"  payload covers {b.tok_in/dash_in:.1%} of dashboard in, "
      f"{b.tok_out/dash_out:.1%} of dashboard out")
assert b.tok_in <= dash_in and b.tok_out <= dash_out
assert b.tok_in / dash_in > 0.95 and b.tok_out / dash_out > 0.95
# small gap expected: dashboard counts every call that day, incl. any unsaved dev calls

# D3. per-question economics (OpenAI batches only)
print(f"\nbatches: $27.76 -> ${27.76/140:.3f} per question, "
      f"${27.76/1680:.4f} per arm-run")
print("reviewed20 production share: ~$4.6-4.9 "
      "(token-proportion estimate of the $5.91 day; exact split not recoverable)")
# note: tokens mix GPT-5.4 and GPT-4.1-mini, so a per-model dollar split
# cannot be derived from these aggregates

# D4. dataset construction (Qwen3Max) - bank records, authoritative $
#     reconstruction detail lives in qwen_usage.ipynb
#     (~1.74M tokens, est. $8.38; bank total slightly higher, monthly billing)
qwen_spend = pd.DataFrame([
    dict(date="2026-05-01", usd=1.45),
    dict(date="2026-06-01", usd=4.11),
    dict(date="2026-07-01", usd=4.50),
])

# D5. project totals
openai_total = openai_spend.usd.sum()
qwen_total   = qwen_spend.usd.sum()
print(f"\nOpenAI core experiment:      ${openai_total:.2f}")
print(f"Qwen dataset construction:   ${qwen_total:.2f}")
print(f"project total:               ${openai_total + qwen_total:.2f}")

cost = pd.concat([
    openai_spend.assign(phase="core_experiment_openai")[["phase", "date", "usd", "scope"]],
    qwen_spend.assign(phase="dataset_construction_qwen",
                      scope="bank charge, monthly billing")[["phase", "date", "usd", "scope"]],
])
cost.to_csv(ANALYSIS_DIR / "step6_cost.csv", index=False)
print("\nsaved: step6_tokens.csv, step6_loop_effort.csv, step6_cost.csv")

15 run files
     group  run  n_questions  n_failed  tok_in  tok_out
    batch1    1           35         0 3693768   173821
    batch1    2           35         0 3631533   168133
    batch1    3           35         0 3720037   169438
    batch2    1           35         0 4372112   203450
    batch2    2           35         0 4705212   220133
    batch2    3           35         0 4140977   201405
    batch3    1           35         0 4362488   205884
    batch3    2           35         0 3846214   189928
    batch3    3           35         0 4382933   216397
    batch4    1           35         0 5480861   195247
    batch4    2           35         0 5423449   207113
    batch4    3           35         0 5668061   180006
reviewed20    1           20         0 3180076   123481
reviewed20    2           20         0 3594939   116820
reviewed20    3           20         0 3020787   107850

all 15 runs: 63,223,447 in / 2,679,106 out
batches 1-4: 53,427,645 in / 2,330,955 out
revi

In [ ]:
# ============================================================
# CELL S7-1 - setup: paths, API, prompts, helpers
# step7: second-judge (GPT-5.4) re-verification of
#   (a) all 606 first-pass REFUTED claim instances (have human labels)
#   (b) 300 sampled first-pass SUPPORTED claims (75 per arm, for kappa)
# judge sees the SAME prompt + SAME passages the pipeline verifier saw.
# reads: results/ payloads, analysis/step1_instances.csv, rag_index meta
# writes: results/step7_judgments.jsonl, analysis/step7_*.csv
# ============================================================
import sys, json, re, random, time
import pandas as pd
from pathlib import Path

sys.path.insert(0, "/content/drive/MyDrive/medical_qa_hallucination/experiments")
from config import *   # EXPERIMENTS, RESULTS_DIR, RESULTS_FINAL, INDEX_DIR, ARMS, SEED

ANALYSIS_DIR = Path(EXPERIMENTS) / "analysis"
OUT_JSONL    = Path(RESULTS_DIR) / "step7_judgments.jsonl"

JUDGE_MODEL        = "gpt-5.4"   # second judge; pipeline verifier was gpt-4.1-mini
N_SUPPORTED_SAMPLE = 300
random.seed(SEED)

import getpass
from openai import OpenAI
client = OpenAI(api_key=getpass.getpass("OpenAI API key: "))

def chat(model, system, user, max_tokens=300):
    # gpt-5.4: max_completion_tokens only, temperature stays at model default
    r = client.chat.completions.create(
        model=model, max_completion_tokens=max_tokens,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}])
    return r.choices[0].message.content

# VERIFY_SYS copied VERBATIM from pipeline CELL 5 - do not edit
VERIFY_SYS = (
    "You verify a single clinical claim against retrieved ESC guideline passages. "
    "Use ONLY the passages provided. Decide:\n"
    "  'supported'    -> a passage clearly supports the claim.\n"
    "  'refuted'      -> a passage clearly contradicts the claim.\n"
    "  'unverifiable' -> the passages do not contain enough to judge.\n"
    "Return ONLY JSON, no fences: "
    '{"verdict": str, "chunk_id": str|null, "rationale": str}. '
    "chunk_id is the id of the single most relevant passage (or null if none). "
    "rationale is ONE short sentence.")

def _parse_json(txt):
    txt = re.sub(r"^```(json)?|```$", "", txt.strip(), flags=re.M).strip()
    if not txt.startswith("{"):
        a, b = txt.find("{"), txt.rfind("}")
        if a != -1 and b != -1 and b > a:
            txt = txt[a:b+1]
    return json.loads(txt)

def _norm(s):
    return " ".join(str(s).lower().strip().strip('"').rstrip(" .;:").split())

# one-call API verification, costs fractions of a cent
_t = chat(JUDGE_MODEL, VERIFY_SYS,
          "Claim:\nMetformin is a guideline recommendation.\n\n"
          "Retrieved passages:\n[test:0001:X:prose:s]\n"
          "Metformin is recommended as first-line therapy.\n\nVerify the claim.")
print("chat OK:", _t[:120])

OpenAI API key: ··········
chat OK: {"verdict":"supported","chunk_id":"test:0001:X:prose:s","rationale":"The passage explicitly states that metformin is rec


In [ ]:
# ============================================================
# CELL S7-2 - extraction: flagged (refuted) + supported pool
# ============================================================
RUN_GROUPS = ["reviewed20", "batch1", "batch2", "batch3", "batch4"]

def find_run_files():
    out = []
    for g in RUN_GROUPS:
        for r in (1, 2, 3):
            pat = re.compile(rf"results_{g}_run{r}(_|\.)")
            cands = [p for d in {Path(RESULTS_FINAL), Path(RESULTS_DIR)}
                     for p in d.glob(f"results_{g}_run{r}*.json") if pat.match(p.name)]
            assert cands, f"no file for {g} run{r}"
            ok = [p for p in cands if "failed" not in p.name]
            out.append((g, r, sorted(ok or cands)[0]))
    return sorted(set(out))

def arm_items(q):
    arms = q["arms"]
    return arms.items() if isinstance(arms, dict) else [(a["arm"], a) for a in arms]

flagged, supported_pool = [], []
FILES = find_run_files()
print(f"{len(FILES)} run files")   # expect 15
for group, run, path in FILES:
    p = json.loads(path.read_text())
    for q in p["results"]:
        for arm_name, a in arm_items(q):
            for v in a["history"][0]["verified"]:   # FIRST PASS ONLY
                rec = dict(uid=q["uid"], qtype=q.get("type"), code=q.get("code"),
                           section_group=q.get("section_group"), group=group, run=run,
                           arm=arm_name, claim=v["claim"], verdict=v["verdict"],
                           retrieved_ids=v.get("retrieved_ids") or [],
                           chunk_id=v.get("chunk_id"),
                           verifier_rationale=v.get("rationale"))
                if v["verdict"] == "refuted":
                    flagged.append(rec)
                elif v["verdict"] == "supported":
                    supported_pool.append(rec)

assert len(flagged) == 606, f"expected 606 flagged, got {len(flagged)}"
print(f"flagged: {len(flagged)}   supported pool: {len(supported_pool)}")
# expect: flagged 606, supported pool 14274

15 run files
flagged: 606   supported pool: 14274


In [ ]:
# ============================================================
# CELL S7-3 - human labels join + stratified supported sample
# ============================================================
inst = pd.read_csv(ANALYSIS_DIR / "step1_instances.csv")
# columns: group, run, arm, uid, text, row_i, sim, manual_fix, label
inst["_key"] = inst.apply(lambda x: (x["uid"], x["run"], x["arm"], _norm(x["text"])), axis=1)
assert inst["_key"].is_unique, "duplicate keys in step1_instances, investigate before joining"
lab    = dict(zip(inst["_key"], inst["label"]))
row_of = dict(zip(inst["_key"], inst["row_i"]))

for rec in flagged:
    k = (rec["uid"], rec["run"], rec["arm"], _norm(rec["claim"]))
    rec["human_label"] = lab.get(k)
    rec["row_i"] = row_of.get(k)
missing = [r for r in flagged if r["human_label"] is None]
assert not missing, f"{len(missing)} unmatched"
print(pd.Series([r["human_label"] for r in flagged]).value_counts())
# expect: true_hallucination 284, verifier_error 235, cross_source 50, version_mismatch 37

# stratified sample: 75 supported claims per arm
random.shuffle(supported_pool)
sample = []
for arm in ARMS:
    pool = [r for r in supported_pool if r["arm"] == arm]
    sample += pool[:N_SUPPORTED_SAMPLE // 4]
assert len(sample) == N_SUPPORTED_SAMPLE
print(f"sampled {len(sample)} supported claims")

true_hallucination    284
verifier_error        235
cross_source           50
version_mismatch       37
Name: count, dtype: int64
sampled 300 supported claims


In [ ]:
# ============================================================
# CELL S7-4 - positional chunk resolver + validation
# id format: {index}:{NNNN}:{code}:{chunk_type}:{section title}
# meta jsonl has no id field; NNNN is the zero-based line index
# ============================================================
META = {}
for name in ("coarse", "atomic"):
    lines = (Path(INDEX_DIR) / f"{name}_meta.jsonl").read_text().splitlines()
    META[name] = [json.loads(l) for l in lines]
    print(f"{name}: {len(META[name])} meta records")

def _parts(rid):
    p = rid.split(":", 4)   # section title may itself contain ':'
    return p[0], int(p[1]), p[2], p[3]

def _valid(rid, off):
    try:
        idx_name, num, code, ctype = _parts(rid)
        rec = META[idx_name][num + off]
        return rec["code"] == code and rec["chunk_type"] == ctype
    except Exception:
        return False

all_ids = [i for r in flagged + sample for i in r["retrieved_ids"]]
ok0 = sum(_valid(i, 0) for i in all_ids)
ok1 = sum(_valid(i, 1) for i in all_ids)
print(f"zero-based: {ok0}/{len(all_ids)}   one-based: {ok1}/{len(all_ids)}")
OFFSET = 0 if ok0 >= ok1 else 1
assert max(ok0, ok1) / len(all_ids) > 0.99, "id mapping unreliable, paste this output"
print(f"OFFSET = {OFFSET}")

def resolve_text(rid):
    idx_name, num, _, _ = _parts(rid)
    return META[idx_name][num + OFFSET]["text"]

def format_passages(retrieved_ids):
    return "\n\n".join(f"[{rid}]\n{resolve_text(rid)}" for rid in retrieved_ids)

print(format_passages(flagged[0]["retrieved_ids"])[:400])   # eyeball: real ESC text

coarse: 525 meta records
atomic: 1035 meta records
zero-based: 4439/4439   one-based: 3821/4439
OFFSET = 0
[coarse:0080:VHD:prose:6.1 Diagnosis and management of]
[VHD · §6.1 Diagnosis and management of]
Diagnosis and management of
coronary artery disease The presence of CAD plays an important role in decision-making regarding the timing and modality of treatment, and should be assessed before Heart Team discussion. In patients with a low or moderate pre-test likelihood of obstructive CAD (≤50%), CCTA 


In [ ]:
# ============================================================
# CELL S7-5 - judging loop (resume-safe, gated, error-proof)
# ============================================================
# cleanup: if a previous run wrote only error rows, remove the file first
if OUT_JSONL.exists():
    _prev = pd.read_json(OUT_JSONL, lines=True)
    if len(_prev) and (_prev.judge_verdict == "error").all():
        OUT_JSONL.unlink()
        print(f"removed {len(_prev)} error-only rows from previous failed run")

todo = [dict(r, kind="flagged") for r in flagged] + [dict(r, kind="supported") for r in sample]

def _key(d):
    return (d["uid"], d["run"], d["arm"], _norm(d["claim"]), d["kind"])

done = set()
if OUT_JSONL.exists():
    for line in OUT_JSONL.open():
        done.add(_key(json.loads(line)))
print(f"already judged: {len(done)} / {len(todo)}")

est_in = sum(len(format_passages(r["retrieved_ids"])) // 4 + 100
             for r in todo if _key(r) not in done)
print(f"to judge: {len(todo) - len(done)} calls, est. input ~{est_in/1e6:.2f}M tokens, "
      f"~${est_in/1e6*2.50:.2f} in at GPT-5.4 list prices")

def judge_one(rec, tries=3):
    user = (f"Claim:\n{rec['claim']}\n\nRetrieved passages:\n"
            f"{format_passages(rec['retrieved_ids'])}\n\nVerify the claim.")
    last = None
    for t in range(tries):
        try:
            v = _parse_json(chat(JUDGE_MODEL, VERIFY_SYS, user))
            return {"judge_verdict": v.get("verdict"), "judge_chunk_id": v.get("chunk_id"),
                    "judge_rationale": v.get("rationale")}
        except Exception as e:
            last = str(e)[:200]
            time.sleep(5 * (t + 1))
    return {"judge_verdict": "error", "judge_chunk_id": None, "judge_rationale": last}

# GATE: smoke test must return a real verdict, otherwise the loop never starts
smoke = next(r for r in todo if _key(r) not in done)
smoke_res = judge_one(smoke)
print("smoke:", {k: smoke[k] for k in ("uid", "arm")}, "->", smoke_res)
assert smoke_res["judge_verdict"] in ("supported", "refuted", "unverifiable"), \
    "smoke test failed, loop NOT started, paste this output"

n_new, consec_err = 0, 0
with OUT_JSONL.open("a") as f:
    for rec in todo:
        if _key(rec) in done:
            continue
        out = judge_one(rec)
        if out["judge_verdict"] == "error":      # never written; resume retries
            consec_err += 1
            print("  error, skipped:", out["judge_rationale"][:100])
            if consec_err >= 10:
                raise RuntimeError("10 consecutive errors, aborting; rerun resumes safely")
            continue
        consec_err = 0
        f.write(json.dumps({**rec, "judge_model": JUDGE_MODEL, **out}) + "\n")
        f.flush()
        n_new += 1
        if n_new % 50 == 0:
            print(f"  judged {n_new} new (total {len(done) + n_new}/{len(todo)})")
print(f"done. new judgments written: {n_new}")

removed 906 error-only rows from previous failed run
already judged: 0 / 906
to judge: 906 calls, est. input ~2.56M tokens, ~$6.41 in at GPT-5.4 list prices
smoke: {'uid': 'VHD_S5_Q4', 'arm': 'coarse'} -> {'judge_verdict': 'refuted', 'judge_chunk_id': 'coarse:0086:VHD:table_whole:Recommendation_Table_1', 'judge_rationale': 'The guideline recommends invasive coronary angiography before valve intervention only for high pre-test likelihood of obstructive CAD or severe ventricular SMR, not broadly for severe VHD with LV systolic dysfunction.'}
  judged 50 new (total 50/906)
  judged 100 new (total 100/906)
  judged 150 new (total 150/906)
  judged 200 new (total 200/906)
  judged 250 new (total 250/906)
  judged 300 new (total 300/906)
  judged 350 new (total 350/906)
  judged 400 new (total 400/906)
  judged 450 new (total 450/906)
  judged 500 new (total 500/906)
  judged 550 new (total 550/906)
  judged 600 new (total 600/906)
  judged 650 new (total 650/906)
  judged 700 new (total 700

In [ ]:
# ============================================================
# CELL S7-6 - analysis: judge vs human labels, agreement, kappa
# writes: analysis/step7_verifier_agreement.csv, step7_flagged_x_judge.csv
# ============================================================
jud = pd.read_json(OUT_JSONL, lines=True)
jud = jud[jud.judge_verdict != "error"]
print(f"judgments: {len(jud)} (errors excluded)")
print(jud.judge_verdict.value_counts())

fl = jud[jud.kind == "flagged"].copy()
su = jud[jud.kind == "supported"].copy()

# T-S7a: human label x judge verdict on the flagged 606
t_a = pd.crosstab(fl.human_label, fl.judge_verdict, margins=True)
print("\nhuman label x judge verdict (flagged):"); print(t_a)
t_a.to_csv(ANALYSIS_DIR / "step7_flagged_x_judge.csv")

def rate(df, label, verdict):
    sub = df[df.human_label == label]
    n = (sub.judge_verdict == verdict).sum()
    return f"{n/len(sub):.1%} ({n}/{len(sub)})" if len(sub) else "n/a"

print("\nTH flags confirmed by judge (judge=refuted): ", rate(fl, "true_hallucination", "refuted"))
print("verifier_error overturned (judge=supported): ", rate(fl, "verifier_error", "supported"))
print("cross_source confirmed (judge=refuted):      ", rate(fl, "cross_source", "refuted"))
print("version_mismatch confirmed (judge=refuted):  ", rate(fl, "version_mismatch", "refuted"))

# binary agreement + Cohen's kappa over all judged claims
jud["pipe_binary"] = (jud.kind == "flagged")
jud["judge_binary"] = jud.judge_verdict == "refuted"
agree = (jud.pipe_binary == jud.judge_binary).mean()
p_a, p_b = jud.pipe_binary.mean(), jud.judge_binary.mean()
p_e = p_a * p_b + (1 - p_a) * (1 - p_b)
kappa = (agree - p_e) / (1 - p_e)
print(f"\nbinary refuted/not-refuted agreement: {agree:.1%}, Cohen's kappa = {kappa:.3f}")

# correction-harm robustness (verifier-error-driven harm rows)
oc = pd.read_csv(ANALYSIS_DIR / "step3_outcome_class_final.csv")
idcol = next((c for c in ("row_i", "row", "idx", "index") if c in oc.columns), None)
if idcol is None:
    oc = oc.reset_index().rename(columns={oc.index.name or "index": "row_i"})
    idcol = "row_i"
harm = set(oc.loc[oc.outcome_class == "correction_harm", idcol])
sub = fl[fl.row_i.isin(harm) & (fl.human_label == "verifier_error")]
print(f"\ncorrection-harm instances judged: {len(sub)} "
      f"(expect 15 rows worth)")
print(sub.judge_verdict.value_counts())

summ = pd.DataFrame([
    dict(metric="TH flags confirmed (judge=refuted)",        value=rate(fl, "true_hallucination", "refuted")),
    dict(metric="verifier_error overturned (judge=supported)", value=rate(fl, "verifier_error", "supported")),
    dict(metric="cross_source confirmed (judge=refuted)",    value=rate(fl, "cross_source", "refuted")),
    dict(metric="version_mismatch confirmed (judge=refuted)",value=rate(fl, "version_mismatch", "refuted")),
    dict(metric="binary agreement (all judged)",             value=f"{agree:.1%}"),
    dict(metric="Cohen's kappa",                             value=f"{kappa:.3f}"),
])
summ.to_csv(ANALYSIS_DIR / "step7_verifier_agreement.csv", index=False)
print("\nsaved: step7_verifier_agreement.csv, step7_flagged_x_judge.csv")

judgments: 906 (errors excluded)
judge_verdict
refuted         394
supported       332
unverifiable    180
Name: count, dtype: int64

human label x judge verdict (flagged):
judge_verdict       refuted  supported  unverifiable  All
human_label                                              
cross_source             21          2            27   50
true_hallucination      234         13            37  284
verifier_error          113         72            50  235
version_mismatch         15          8            14   37
All                     383         95           128  606

TH flags confirmed by judge (judge=refuted):  82.4% (234/284)
verifier_error overturned (judge=supported):  30.6% (72/235)
cross_source confirmed (judge=refuted):       42.0% (21/50)
version_mismatch confirmed (judge=refuted):   40.5% (15/37)

binary refuted/not-refuted agreement: 74.2%, Cohen's kappa = 0.505

correction-harm instances judged: 44 (expect 15 rows worth)
judge_verdict
refuted         24
supported      

In [ ]:
# ============================================================
# CELL S7-7 (v2) - export TWO expert workbooks
#   WB1 accepted-claims probe: 89 random (judge-agreed) + 11 judge-disputed
#   WB2 TH-disputed: the 50 TH-labeled claims the judge did NOT confirm
# both blinded (no model info), shuffled, dropdowns, instructions sheet
# writes: audit/supported_probe/wb1_accepted_probe_sent_<date>.xlsx
#         audit/supported_probe/wb2_th_disputed_sent_<date>.xlsx
#         analysis/step7_fn_probe_internal.csv (has stratum + verdicts)
#         analysis/step7_th_disputed_internal.csv
# ============================================================
from datetime import date
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

PROBE_DIR = Path(EXPERIMENTS) / "audit" / "supported_probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)
TODAY = date.today().isoformat()

jud = pd.read_json(OUT_JSONL, lines=True)
jud = jud[jud.judge_verdict != "error"].copy()
jud["guideline"] = jud.uid.str.split("_").str[0]

qtext = {}
for group, run, path in FILES:
    p = json.loads(path.read_text())
    for q in p["results"]:
        qtext[q["uid"]] = q["question"]
jud["question"] = jud.uid.map(qtext)

# ---------- WB1: accepted-claims probe ----------
su = jud[jud.kind == "supported"].copy()
disp = su[su.judge_verdict == "refuted"]                 # expect 11
pool = su[su.judge_verdict != "refuted"]                 # expect 289
assert len(disp) == 11 and len(pool) == 289, (len(disp), len(pool))

N_A = {"coarse": 22, "atomic": 22, "atomic_gl_filter": 22, "atomic_sec_filter": 23}
rand = pd.concat([pool[pool.arm == a].sample(n=n, random_state=SEED)
                  for a, n in N_A.items()])
rand["stratum"] = "random"
disp["stratum"] = "judge_disputed"
wb1 = pd.concat([rand, disp]).sample(frac=1, random_state=SEED).reset_index(drop=True)
wb1.insert(0, "item_no", range(1, len(wb1) + 1))
assert len(wb1) == 100
wb1.to_csv(ANALYSIS_DIR / "step7_fn_probe_internal.csv", index=False)
print("WB1:", len(wb1), "rows |", wb1.stratum.value_counts().to_dict(), "|",
      wb1.arm.value_counts().to_dict())

# ---------- WB2: TH-disputed ----------
fl = jud[jud.kind == "flagged"].copy()
wb2 = fl[(fl.human_label == "true_hallucination") & (fl.judge_verdict != "refuted")] \
        .sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(wb2) == 50, len(wb2)
wb2.insert(0, "item_no", range(1, len(wb2) + 1))
wb2.to_csv(ANALYSIS_DIR / "step7_th_disputed_internal.csv", index=False)
print("WB2:", len(wb2), "rows | judge said:", wb2.judge_verdict.value_counts().to_dict())

# ---------- workbook builder ----------
def make_workbook(path, df, extra_note):
    wb = Workbook()
    ins = wb.active
    ins.title = "instructions"
    lines = [
        "Task: clinical correctness check of statements",
        "",
        "Each row in the 'review' sheet is one statement taken from an AI-generated",
        "answer to the question shown beside it. For every row, please judge ONLY",
        "whether the statement is clinically correct according to current ESC",
        "guideline knowledge, in the context of that question.",
        "",
        extra_note,
        "",
        "verdict options (column F, dropdown):",
        "  correct   - the statement is clinically right",
        "  incorrect - the statement is clinically wrong (wrong threshold, wrong drug,",
        "              wrong class of recommendation, wrong population, etc.)",
        "  unclear   - you cannot judge from the information given",
        "",
        "Column G (comment) is optional but very welcome for 'incorrect' rows.",
        "The rows are in random order: if you cannot finish, completing any part helps.",
        "Thank you very much!",
    ]
    for i, t in enumerate(lines, 1):
        ins.cell(row=i, column=1, value=t)
    ins.column_dimensions["A"].width = 72

    ws = wb.create_sheet("review")
    ws.append(["item_no", "guideline", "uid", "question", "claim",
               "expert_verdict", "expert_comment"])
    for c in ws[1]:
        c.font = Font(bold=True)
        c.fill = PatternFill("solid", fgColor="DDE8F0")
    for _, r in df.iterrows():
        ws.append([r.item_no, r.guideline, r.uid, r.question, r.claim, "", ""])
    for col, w in zip("ABCDEFG", (8, 10, 16, 60, 60, 16, 40)):
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "A2"
    dv = DataValidation(type="list", formula1='"correct,incorrect,unclear"',
                        allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"F2:F{ws.max_row}")
    wb.save(path)
    print("saved:", path)

make_workbook(PROBE_DIR / f"wb1_accepted_probe_sent_{TODAY}.xlsx", wb1,
              "Some of these statements may be wrong; please judge each one on its own.")
make_workbook(PROBE_DIR / f"wb2_th_disputed_sent_{TODAY}.xlsx", wb2,
              "Many of these statements may be wrong; please judge each one on its own.")

/tmp/ipykernel_1407/2112341650.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  disp["stratum"] = "judge_disputed"


WB1: 100 rows | {'random': 89, 'judge_disputed': 11} | {'coarse': 27, 'atomic_sec_filter': 26, 'atomic_gl_filter': 24, 'atomic': 23}
WB2: 50 rows | judge said: {'unverifiable': 37, 'supported': 13}
saved: /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/supported_probe/wb1_accepted_probe_sent_2026-11-01.xlsx
saved: /content/drive/MyDrive/medical_qa_hallucination/experiments/audit/supported_probe/wb2_th_disputed_sent_2026-11-01.xlsx


In [ ]:
# S7-VIEW: inspect the non-random claims sent to the expert in WB1 and WB2
import os, json, glob, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

sys.path.insert(0, "/content/drive/MyDrive/medical_qa_hallucination/experiments")
from config import DATASET_FINAL, RESULTS_DIR, RESULTS_FINAL, EXPERIMENTS

import pandas as pd
from pathlib import Path

try:
    ANALYSIS_DIR
except NameError:
    from config import EXPERIMENTS
    ANALYSIS_DIR = Path(EXPERIMENTS) / "analysis"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

def _pick(df, *cands):
    """Return the first column name that exists in df."""
    for c in cands:
        if c in df.columns:
            return c
    return None

def show(df, label):
    claim_col = _pick(df, "claim", "text", "claim_text")
    cols = [c for c in [
        _pick(df, "item_no", "item", "no"),
        "group", "run", "arm", "uid",
        claim_col, "human_label", "judge_verdict",
        _pick(df, "stratum"),
    ] if c is not None and c in df.columns]
    out = df[cols].reset_index(drop=True)
    if claim_col and claim_col != "claim":
        out = out.rename(columns={claim_col: "claim"})
    print(f"=== {label}: {len(out)} rows ===")
    display(out)

# ---------- WB1: the judge-disputed claims added on top of the 89 random ----------
wb1 = pd.read_csv(ANALYSIS_DIR / "step7_fn_probe_internal.csv")
print("WB1 strata counts:")
print(wb1["stratum"].value_counts(), "\n")
wb1_extra = wb1[wb1["stratum"] != "random"]
show(wb1_extra, "WB1 judge-disputed claims added to the random 89")

# ---------- WB2: split of the 50 TH-disputed rows by judge verdict ----------
wb2 = pd.read_csv(ANALYSIS_DIR / "step7_th_disputed_internal.csv")
print("\nWB2 judge verdict counts:")
print(wb2["judge_verdict"].value_counts(), "\n")
show(wb2[wb2["judge_verdict"] == "supported"],
     "WB2 TH rows the judge called 'supported'")
show(wb2[wb2["judge_verdict"] == "unverifiable"],
     "WB2 TH rows the judge called 'unverifiable'")

WB1 strata counts:
stratum
random            89
judge_disputed    11
Name: count, dtype: int64 

=== WB1 judge-disputed claims added to the random 89: 11 rows ===


,item_no,group,run,arm,uid,claim,human_label,judge_verdict,stratum
0,15,batch2,2,coarse,CM_S8_Q1,Low-risk features include mildly reduced LV systolic function with LVEF 45–50%.,NaN,refuted,judge_disputed
1,28,batch3,1,atomic_gl_filter,CVD_S4-7_Q1,"In treated adults aged 70 years or older, the ultimate office systolic BP target is 130–139 mmHg.",NaN,refuted,judge_disputed
2,35,batch3,2,coarse,AF_S3_Q3,An AHRE is typically defined by an atrial rate threshold of at least 175 beats per minute lasting at least 5 minutes.,NaN,refuted,judge_disputed
3,44,batch3,2,atomic_gl_filter,VHD_S14_Q5,Therapeutic low molecular weight heparin (LMWH) is an alternative bridging option only when timing is less critical and renal function is adequate.,NaN,refuted,judge_disputed
4,60,batch3,3,coarse,CM_S6-10-6-11_Q1,"The exception to CHA₂DS₂-VASc-based decision-making is atrial fibrillation or flutter in HCM, amyloidosis, and restrictive cardiomyopathy, where anticoagulation is universal.",NaN,refuted,judge_disputed
5,62,batch1,1,atomic_sec_filter,AF_S9_Q5,A reduced DOAC dose should be considered only during the period of concomitant antiplatelet therapy to mitigate bleeding risk.,NaN,refuted,judge_disputed
6,70,batch3,1,atomic_sec_filter,CM_S6-10-6-11_Q1,"For dilated cardiomyopathy (DCM), arrhythmogenic right ventricular cardiomyopathy (ARVC/ACM), left ventricular non-compaction (LVNC), and other cardiomyopathy phenotypes, anticoagulation is guided by the usual CHA₂DS₂-VASc thresholds.",NaN,refuted,judge_disputed
7,75,batch2,2,atomic_sec_filter,AF_S9_Q8,"Triple therapy is not possible without aspirin, so the default is a DOAC-based regimen plus clopidogrel 75 mg daily, or a VKA if a DOAC is unsuitable.",NaN,refuted,judge_disputed
8,79,batch3,3,atomic,AF_S3_Q3,"On implanted cardiac devices, an atrial high-rate episode is typically defined as an atrial rate of at least 175 beats per minute lasting at least 5 minutes.",NaN,refuted,judge_disputed
9,91,batch4,3,coarse,VHD_S8_Q7,"In women, an aortic valve calcium score greater than 1600 Agatston units strongly supports severe aortic stenosis.",NaN,refuted,judge_disputed



WB2 judge verdict counts:
judge_verdict
unverifiable    37
supported       13
Name: count, dtype: int64 

=== WB2 TH rows the judge called 'supported': 13 rows ===


,item_no,group,run,arm,uid,claim,human_label,judge_verdict
0,8,batch4,1,atomic_sec_filter,VHD_S8_Q4,"Normal-flow, low-gradient AS is characterized by stroke volume index (SVI) greater than or equal to 35 mL/m².",true_hallucination,supported
1,12,batch2,1,atomic,CM_S7-1_Q2,"A primary-prevention ICD would be considered on the basis of the LV apical aneurysm, particularly with transmural/extensive LGE and recent unexplained syncope, even if the numerical risk score alone appears only intermediate or low.",true_hallucination,supported
2,16,batch2,1,atomic_sec_filter,CM_S7-1_Q2,"A primary-prevention ICD would be considered on the basis of the LV apical aneurysm, particularly with transmural/extensive LGE and recent unexplained syncope, even if the numerical risk score alone appears only intermediate or low.",true_hallucination,supported
3,23,batch4,1,atomic_gl_filter,VHD_S8_Q4,"Normal-flow, low-gradient AS is characterized by stroke volume index (SVI) greater than or equal to 35 mL/m².",true_hallucination,supported
4,24,batch4,2,atomic,VHD_S8_Q4,"Normal-flow, low-gradient AS is defined by stroke volume index (SVI) ≥35 mL/m².",true_hallucination,supported
5,26,batch1,2,atomic_gl_filter,CM_S12_Q1,Hypertension maps predominantly to hypertrophic cardiomyopathy expression.,true_hallucination,supported
6,30,batch2,1,atomic_gl_filter,CM_S7-1_Q2,"A primary-prevention ICD would be considered on the basis of the LV apical aneurysm, particularly with transmural/extensive LGE and recent unexplained syncope, even if the numerical risk score alone appears only intermediate or low.",true_hallucination,supported
7,34,batch2,1,coarse,AF_S3_Q4,"The incidence of atrial flutter rises to roughly 5 per 1,000 person-years in people older than 50 years.",true_hallucination,supported
8,36,batch2,1,coarse,AF_S10_Q3,Atrial fibrillation affects about 2–4% of the global population.,true_hallucination,supported
9,38,batch4,2,atomic_gl_filter,VHD_S8_Q4,"Normal-flow, low-gradient AS is defined by stroke volume index (SVI) ≥35 mL/m².",true_hallucination,supported


=== WB2 TH rows the judge called 'unverifiable': 37 rows ===


,item_no,group,run,arm,uid,claim,human_label,judge_verdict
0,1,batch2,2,atomic_gl_filter,CM_S7-1_Q3,"An apical aneurysm should weigh strongly toward primary-prevention ICD discussion, particularly if accompanied by nonsustained ventricular tachycardia, unexplained syncope, extensive late gadolinium enhancement, family history of sudden cardiac death, or left ventricular systolic impairment.",true_hallucination,unverifiable
1,2,batch4,3,coarse,CVD_S3-3_Q1,"For people of West Asian ancestry, the recommended cardiovascular risk multiplier is 1.2.",true_hallucination,unverifiable
2,3,batch4,1,atomic,CVD_S3-3_Q1,"For people of West Asian ancestry, the recommended cardiovascular risk multiplier is 1.2.",true_hallucination,unverifiable
3,4,reviewed20,2,coarse,VHD_S8_Q9,The true hemodynamic burden of aortic stenosis is the maximal Doppler value obtained from any acoustic window.,true_hallucination,unverifiable
4,5,batch3,2,atomic_gl_filter,AF_S7_Q6,Dronedarone is appropriate for patients with normal or mildly impaired left ventricular function and should not be used in patients with heart failure or permanent atrial fibrillation.,true_hallucination,unverifiable
5,6,reviewed20,3,atomic_gl_filter,VHD_S8_Q9,The true hemodynamic severity of aortic stenosis is best represented by the highest velocity and gradient measurements obtained.,true_hallucination,unverifiable
6,7,batch4,1,coarse,CM_S7-4_Q1,"If symptoms or ventricular arrhythmia burden persist despite beta-blockade in ARVC, escalation to sotalol is appropriate.",true_hallucination,unverifiable
7,9,batch4,1,atomic_sec_filter,CVD_S3-3_Q1,"For people of West Asian ancestry, the recommended cardiovascular risk multiplier is 1.2.",true_hallucination,unverifiable
8,10,batch3,3,coarse,AF_S5_Q3,An isolated systolic blood pressure value of 108 mmHg is acceptable in a treated patient with atrial fibrillation if there are no symptoms.,true_hallucination,unverifiable
9,11,batch2,2,atomic,AF_S9_Q2,"In inherited cardiomyopathies, class Ic antiarrhythmic agents such as flecainide and propafenone are generally avoided because structural myocardial disease increases proarrhythmic risk.",true_hallucination,unverifiable


In [ ]:
# ============================================================
# CELL S7-9: WB1 expert probe analysis
# Stratified miss-rate estimate for the verifier-accepted pool
# WB1 = 89 random (of 289 judge-agreed) + 11 judge-disputed
# ============================================================
import os, json, glob, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, "/content/drive/MyDrive/medical_qa_hallucination/experiments")
from config import DATASET_FINAL, RESULTS_DIR, RESULTS_FINAL, EXPERIMENTS


try:
    ANALYSIS_DIR
except NameError:
    from config import EXPERIMENTS
    ANALYSIS_DIR = Path(EXPERIMENTS) / "analysis"

PROBE_DIR = Path(EXPERIMENTS) / "audit" / "supported_probe"
RET_FILE  = PROBE_DIR / "wb1_accepted_probe_returned_2026-11-03.xlsx"
INT_FILE  = ANALYSIS_DIR / "step7_fn_probe_internal.csv"

N_AGREED, N_DISPUTED = 289, 11        # population strata (from S7-7)
N_ACCEPTED_POOL = 14274               # total verifier-accepted claims (S7-2)
SEED = 42

# ---------- load + join ----------
ret = pd.read_excel(RET_FILE, sheet_name="review")
internal = pd.read_csv(INT_FILE)

idcol = next(c for c in ["item_no", "item", "no"] if c in internal.columns)
internal = internal.rename(columns={idcol: "item_no"})
assert internal["item_no"].is_unique and len(internal) == 100, "internal map must have 100 unique item_no"

ret["expert_verdict"] = ret["expert_verdict"].astype(str).str.strip().str.lower()
assert set(ret["expert_verdict"]) <= {"correct", "incorrect", "unclear"}, \
    f"unexpected verdicts: {set(ret['expert_verdict'])}"

m = ret.merge(internal[["item_no", "stratum", "judge_verdict"]], on="item_no", how="left", validate="1:1")
assert m["stratum"].notna().all(), "some returned rows did not match the internal map"
m["is_random"] = m["stratum"].astype(str).str.lower().str.startswith("rand")
m["incorrect"] = (m["expert_verdict"] == "incorrect").astype(int)
m = m[m["expert_verdict"] != "unclear"]          # exclude 'unclear' from rate base

print(f"strata in returned file: {m['stratum'].value_counts().to_dict()}")
print(f"judge verdicts among the 5 incorrect: "
      f"{m.loc[m['incorrect']==1, 'judge_verdict'].value_counts().to_dict()}\n")

# ---------- stratified rates ----------
def wilson(x, n, z=1.959964):
    if n == 0: return (np.nan, np.nan)
    p = x / n; d = 1 + z*z/n
    c = p + z*z/(2*n); mm = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c-mm)/d, (c+mm)/d

vr = m.loc[m["is_random"], "incorrect"].to_numpy()
vd = m.loc[~m["is_random"], "incorrect"].to_numpy()
xr, nr = int(vr.sum()), len(vr)
xd, nd = int(vd.sum()), len(vd)

rate_r, rate_d = xr/nr, xd/nd
rate_pop = rate_r * (N_AGREED/300) + rate_d * (N_DISPUTED/300)

rng = np.random.default_rng(SEED)
boot = []
for _ in range(10000):
    br = rng.choice(vr, size=nr, replace=True).mean() if nr else 0.0
    bd = rng.choice(vd, size=nd, replace=True).mean() if nd else 0.0
    boot.append(br*(N_AGREED/300) + bd*(N_DISPUTED/300))
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])

print(f"random stratum:          {xr}/{nr} incorrect = {rate_r:.1%}  (Wilson 95% CI "
      f"{wilson(xr,nr)[0]:.1%}..{wilson(xr,nr)[1]:.1%})")
print(f"judge-disputed stratum:  {xd}/{nd} incorrect = {rate_d:.1%}  (Wilson 95% CI "
      f"{wilson(xd,nd)[0]:.1%}..{wilson(xd,nd)[1]:.1%})")
print(f"population miss-rate (weighted): {rate_pop:.1%}  (bootstrap 95% CI {ci_lo:.1%}..{ci_hi:.1%})")
print(f"applied to {N_ACCEPTED_POOL} accepted claims: "
      f"~{rate_pop*N_ACCEPTED_POOL:.0f} potentially incorrect "
      f"({ci_lo*N_ACCEPTED_POOL:.0f}..{ci_hi*N_ACCEPTED_POOL:.0f})")

# ---------- save ----------
out = pd.DataFrame([
    dict(stratum="random",         n=nr, incorrect=xr, rate=rate_r,
         ci_lo=wilson(xr,nr)[0], ci_hi=wilson(xr,nr)[1]),
    dict(stratum="judge_disputed", n=nd, incorrect=xd, rate=rate_d,
         ci_lo=wilson(xd,nd)[0], ci_hi=wilson(xd,nd)[1]),
    dict(stratum="population_weighted", n=nr+nd, incorrect=xr+xd, rate=rate_pop,
         ci_lo=ci_lo, ci_hi=ci_hi),
])
out.to_csv(ANALYSIS_DIR / "step7_fn_probe_results.csv", index=False)
m.to_csv(ANALYSIS_DIR / "step7_fn_probe_merged.csv", index=False)
print(f"\nsaved: {ANALYSIS_DIR/'step7_fn_probe_results.csv'}")
print(f"saved: {ANALYSIS_DIR/'step7_fn_probe_merged.csv'}")

# ---------- thesis-ready ----------
print(f"""
THESIS-READY:
Of 100 verifier-accepted claims reviewed blind by the clinical expert
(89 random + 11 judge-disputed), {xr+xd} were judged clinically incorrect
({(xr+xd)/(nr+nd):.0%} correctness of the accepted pool). Stratified
estimation gives a population miss-rate of {rate_pop:.1%} (95% CI
{ci_lo:.1%}..{ci_hi:.1%}), i.e., an estimated {rate_pop*N_ACCEPTED_POOL:.0f}
of {N_ACCEPTED_POOL} pipeline-accepted claims may be clinically incorrect.
All identified errors were boundary-level deviations (numeric thresholds or
recommendation-strength class), not fabricated clinical content.
""")

strata in returned file: {'random': 89, 'judge_disputed': 11}
judge verdicts among the 5 incorrect: {'refuted': 3, 'supported': 1, 'unverifiable': 1}

random stratum:          2/89 incorrect = 2.2%  (Wilson 95% CI 0.6%..7.8%)
judge-disputed stratum:  3/11 incorrect = 27.3%  (Wilson 95% CI 9.7%..56.6%)
population miss-rate (weighted): 3.2%  (bootstrap 95% CI 0.7%..6.7%)
applied to 14274 accepted claims: ~452 potentially incorrect (95..963)

saved: /content/drive/MyDrive/medical_qa_hallucination/experiments/analysis/step7_fn_probe_results.csv
saved: /content/drive/MyDrive/medical_qa_hallucination/experiments/analysis/step7_fn_probe_merged.csv

THESIS-READY:
Of 100 verifier-accepted claims reviewed blind by the clinical expert
(89 random + 11 judge-disputed), 5 were judged clinically incorrect
(5% correctness of the accepted pool). Stratified
estimation gives a population miss-rate of 3.2% (95% CI
0.7%..6.7%), i.e., an estimated 452
of 14274 pipeline-accepted claims may be clinically inco

In [ ]:
# ============================================================
# CELL S7-8 v2: robustness of the Step 7 headline numbers
# Offline: reads step7_judgments.jsonl (human_label included).
# Adds: bootstrap CIs (agreement, kappa), per-arm and
# per-guideline TH-confirmation and VE-rejection rates.
# Saves: analysis/step7_robustness.csv
# ============================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

try:
    OUT_JSONL; ANALYSIS_DIR
except NameError:
    from config import EXPERIMENTS, RESULTS_DIR
    ANALYSIS_DIR = Path(EXPERIMENTS) / "analysis"
    OUT_JSONL = Path(RESULTS_DIR) / "step7_judgments.jsonl"

SEED, B = 42, 5000

# ---------- load judgments ----------
rows = [json.loads(l) for l in open(OUT_JSONL, encoding="utf-8") if l.strip()]
J = pd.DataFrame(rows)
assert len(J) == 906, f"expected 906 judgments, got {len(J)}"
print("kind counts:", J["kind"].value_counts().to_dict())

J["verifier_bin"] = (J["kind"] == "flagged").astype(int)   # flagged = verifier said refuted
J["judge_bin"] = (J["judge_verdict"] == "refuted").astype(int)
J["guideline"] = J["uid"].str.extract(r"^(VHD|AF|CM|CVD)")
assert J["guideline"].notna().all(), "uid prefix did not match any guideline"

# ---------- flagged subset: human labels already in the JSONL ----------
F = J[J["verifier_bin"] == 1].copy()
assert len(F) == 606, f"expected 606 flagged, got {len(F)}"
lc = F["human_label"].value_counts().to_dict()
print("human label counts:", lc)
assert lc.get("true_hallucination") == 284 and lc.get("verifier_error") == 235 \
   and lc.get("cross_source") == 50 and lc.get("version_mismatch") == 37, \
    "human label counts do not reconcile with the frozen audit (284/235/50/37)"

# ---------- reproduction gate (locked numbers) ----------
agree = float((J["verifier_bin"] == J["judge_bin"]).mean())
kappa = float(cohen_kappa_score(J["verifier_bin"], J["judge_bin"]))
print(f"\nreproduction check: n={len(J)}, agreement={agree:.3f}, kappa={kappa:.3f}")
assert abs(agree - 0.742) < 0.005 and abs(kappa - 0.505) < 0.01, \
    "reproduction vs locked numbers (74.2%, 0.505) FAILED; stop and diagnose"

# ---------- bootstrap CIs ----------
rng = np.random.default_rng(SEED)
v, j = J["verifier_bin"].to_numpy(), J["judge_bin"].to_numpy()
idx = np.arange(len(J))
ba = np.empty(B); bk = np.empty(B)
for b in range(B):
    s = rng.choice(idx, size=len(idx), replace=True)
    ba[b] = (v[s] == j[s]).mean()
    bk[b] = cohen_kappa_score(v[s], j[s])
ag_lo, ag_hi = np.percentile(ba, [2.5, 97.5])
k_lo, k_hi = np.percentile(bk, [2.5, 97.5])

# ---------- breakdowns ----------
def wilson(x, n, z=1.959964):
    if n == 0: return (np.nan, np.nan)
    p = x / n; d = 1 + z*z/n
    c = p + z*z/(2*n); m = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c-m)/d, (c+m)/d

out = [dict(section="overall", level="all", metric="binary_agreement", n=len(J),
            value=agree, ci_lo=ag_lo, ci_hi=ag_hi),
       dict(section="overall", level="all", metric="cohens_kappa", n=len(J),
            value=kappa, ci_lo=k_lo, ci_hi=k_hi)]

th = F[F["human_label"] == "true_hallucination"]
ve = F[F["human_label"] == "verifier_error"]
for sec, col in [("by_arm", "arm"), ("by_guideline", "guideline")]:
    for lvl in sorted(F[col].unique()):
        t = th[th[col] == lvl]; e = ve[ve[col] == lvl]
        x1, n1 = int(t["judge_bin"].sum()), len(t)
        x0, n0 = int((1 - e["judge_bin"]).sum()), len(e)
        out.append(dict(section=sec, level=lvl, metric="TH_confirmation_rate", n=n1,
                        value=x1/n1 if n1 else np.nan,
                        ci_lo=wilson(x1, n1)[0], ci_hi=wilson(x1, n1)[1]))
        out.append(dict(section=sec, level=lvl, metric="VE_rejection_rate", n=n0,
                        value=x0/n0 if n0 else np.nan,
                        ci_lo=wilson(x0, n0)[0], ci_hi=wilson(x0, n0)[1]))

out = pd.DataFrame(out)
pd.set_option("display.width", 200); pd.set_option("display.max_rows", None)
print(out.to_string(index=False,
      formatters={"value": "{:.3f}".format, "ci_lo": "{:.3f}".format, "ci_hi": "{:.3f}".format}))
out.to_csv(ANALYSIS_DIR / "step7_robustness.csv", index=False)
print(f"\nsaved: {ANALYSIS_DIR / 'step7_robustness.csv'}")

kind counts: {'flagged': 606, 'supported': 300}
human label counts: {'true_hallucination': 284, 'verifier_error': 235, 'cross_source': 50, 'version_mismatch': 37}

reproduction check: n=906, agreement=0.742, kappa=0.505
     section             level               metric   n value ci_lo ci_hi
     overall               all     binary_agreement 906 0.742 0.713 0.770
     overall               all         cohens_kappa 906 0.505 0.456 0.554
      by_arm            atomic TH_confirmation_rate  72 0.819 0.715 0.891
      by_arm            atomic    VE_rejection_rate  67 0.463 0.349 0.581
      by_arm  atomic_gl_filter TH_confirmation_rate  71 0.789 0.680 0.868
      by_arm  atomic_gl_filter    VE_rejection_rate  62 0.484 0.364 0.606
      by_arm atomic_sec_filter TH_confirmation_rate  78 0.846 0.750 0.910
      by_arm atomic_sec_filter    VE_rejection_rate  48 0.688 0.547 0.801
      by_arm            coarse TH_confirmation_rate  63 0.841 0.732 0.911
      by_arm            coarse    VE_rej

In [ ]:
# ============================================================
# CELL S6-COST-UPDATE v2: add Step 7 actual cost to step6_cost.csv
# Dashboard 2026-10-31: 910 requests, 2.562M input tokens, $6.34
# Schema: phase, date, usd, scope. Sorted by date, old to new.
# ============================================================
import pandas as pd
from pathlib import Path

try:
    ANALYSIS_DIR
except NameError:
    from config import EXPERIMENTS
    ANALYSIS_DIR = Path(EXPERIMENTS) / "analysis"

COST = ANALYSIS_DIR / "step6_cost.csv"
df = pd.read_csv(COST, dtype={"phase": str, "scope": str}).fillna("")
assert list(df.columns) == ["phase", "date", "usd", "scope"], f"unexpected schema: {df.columns.tolist()}"

# drop any previously appended Step 7 row (idempotent re-runs)
df = df[df["phase"] != "step7_second_verifier_openai"]

step7 = dict(phase="step7_second_verifier_openai",
             date="2026-10-31",
             usd=6.34,
             scope="second-judge re-verification: 906 judgments (606 flagged + 300 supported); 910 requests, 2.562M input tokens; pre-run estimate 2.56M tok / $6.41")
df = pd.concat([df, pd.DataFrame([step7])], ignore_index=True)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df["date"] = df["date"].dt.strftime("%Y-%m-%d")
df.to_csv(COST, index=False)

print(df.to_string(index=False))
print(f"\nsaved: {COST}")
print("\nreconciled totals:")
print(f"  OpenAI (pipeline + Step 7): ${1.75 + 5.91 + 27.76 + 6.34:.2f}")
print(f"  Qwen3-Max (dataset construction): ${1.45 + 4.11 + 4.50:.2f}")
print(f"  project total: ${1.75 + 5.91 + 27.76 + 6.34 + 1.45 + 4.11 + 4.50:.2f}")

                       phase       date   usd                                                                                                                                            scope
   dataset_construction_qwen 2026-05-01  1.45                                                                                                                     bank charge, monthly billing
   dataset_construction_qwen 2026-06-01  4.11                                                                                                                     bank charge, monthly billing
      core_experiment_openai 2026-06-17  1.75                                                                                                                                      dev + pilot
      core_experiment_openai 2026-06-29  5.91                                                                                                reviewed20 round (3 runs x 4 arms) + dev same day
   dataset_construction_qwen 2026-07-01  4.50

In [ ]:
# ============================================================
# CELL S8-FREEZE v2: freeze manifest (reproducibility appendix)
# Hashes all frozen inputs; records environment versions.
# Saves: analysis/freeze_manifest.csv + analysis/FREEZE.md
# Run LAST, after S7-8 and the cost update.
# ============================================================
import hashlib, sys, importlib.metadata as md
from pathlib import Path
from datetime import date
import pandas as pd

from config import EXPERIMENTS
EXPERIMENTS = Path(EXPERIMENTS)
ANALYSIS_DIR = EXPERIMENTS / "analysis"
ROOT = Path("/content/drive/MyDrive/medical_qa_hallucination")

hits = list(ROOT.rglob("dataset_reviewed.jsonl"))
assert len(hits) == 1, f"expected exactly one dataset_reviewed.jsonl, found: {hits}"
DATA_DIR = hits[0].parent
print(f"dataset dir: {DATA_DIR.relative_to(ROOT)}")
FROZEN_DIRS = [DATA_DIR, EXPERIMENTS / "results", EXPERIMENTS / "audit"]

# ---------- hash all frozen input files ----------
recs = []
for d in FROZEN_DIRS:
    if not d.exists():
        print(f"WARNING missing dir: {d}")
        continue
    for p in sorted(d.rglob("*")):
        if p.is_file():
            recs.append(dict(path=str(p.relative_to(ROOT)),
                             bytes=p.stat().st_size,
                             sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
m = pd.DataFrame(recs)
m.to_csv(ANALYSIS_DIR / "freeze_manifest.csv", index=False)

# ---------- reconciliation checks ----------
n_runs = sum(1 for r in recs if Path(r["path"]).name.startswith("results_") and r["path"].endswith(".json"))
judg = [r for r in recs if r["path"].endswith("step7_judgments.jsonl")]
n_judg = None
if judg:
    n_judg = sum(1 for _ in open(ROOT / judg[0]["path"], encoding="utf-8") if _.strip())
print(f"hashed {len(m)} files, {m['bytes'].sum()/1e6:.1f} MB total")
print(f"run payloads found: {n_runs} (expect 15)")
print(f"step7 judgments lines: {n_judg} (expect 906)")
assert n_runs == 15, f"expected 15 run payloads, found {n_runs}"
assert n_judg == 906, f"expected 906 judgments, found {n_judg}"

# ---------- environment ----------
pkgs = ["openai", "pandas", "numpy", "scikit-learn", "faiss-cpu",
        "sentence-transformers", "transformers", "torch", "openpyxl"]
vers = {}
for p in pkgs:
    try:
        vers[p] = md.version(p)
    except md.PackageNotFoundError:
        vers[p] = "not installed"

# ---------- write FREEZE.md ----------
lines = [
    "# Experiment freeze manifest",
    "",
    f"Freeze date: {date.today().isoformat()}",
    f"Python: {sys.version.split()[0]}",
    "",
    "## Environment",
    "",
] + [f"- {p}: {v}" for p, v in vers.items()] + [
    "",
    "## Frozen inputs (hashed in freeze_manifest.csv)",
    "",
]
for d in FROZEN_DIRS:
    sub = m[m["path"].str.startswith(str(d.relative_to(ROOT)))]
    lines.append(f"- {d.relative_to(ROOT)}/: {len(sub)} files, {sub['bytes'].sum()/1e6:.1f} MB")
lines += [
    "",
    "## Reconciliation at freeze",
    "",
    f"- run payloads (results_*.json): {n_runs} (expected 15)",
    f"- step7_judgments.jsonl lines: {n_judg} (expected 906)",
    "- dataset: combined/dataset_reviewed.jsonl (160 questions, locked)",
    "- audit labels: post-2026-10-07 reconciliation (59 TH rows / 284 instances)",
    "",
    "## Notebooks of record",
    "",
    "- pipeline_final.ipynb (experiment runs)",
    "- analysis.ipynb (Steps 1-6, cells B1-B6; Step 7 cells S7-1 to S7-9; robustness S7-8)",
    "- qwen_usage.ipynb (dataset-generation cost accounting)",
]
(ANALYSIS_DIR / "FREEZE.md").write_text("\n".join(lines), encoding="utf-8")
print(f"\nsaved: {ANALYSIS_DIR / 'freeze_manifest.csv'}")
print(f"saved: {ANALYSIS_DIR / 'FREEZE.md'}")

dataset dir: dataset
hashed 1435 files, 384.9 MB total
run payloads found: 15 (expect 15)
step7 judgments lines: 906 (expect 906)

saved: /content/drive/MyDrive/medical_qa_hallucination/experiments/analysis/freeze_manifest.csv
saved: /content/drive/MyDrive/medical_qa_hallucination/experiments/analysis/FREEZE.md


In [ ]:
# A-TYPES: distribution of hallucination error types over audited claims
import pandas as pd, re
from pathlib import Path
cand = list(Path(".").rglob("audit_claims.csv"))
assert cand, "audit_claims.csv not found"
f = cand[0]
print("reading:", f)
df = pd.read_csv(f)
print("shape:", df.shape)
print("columns:", list(df.columns))
pat = re.compile(r"type|fabricat|threshold|scope|strength|class_level", re.I)
cands = [c for c in df.columns if pat.search(c)]
print("candidate type columns:", cands)
col = cands[0]
sub = df[df[col].notna()]
print("\nrows with a type label:", len(sub))
print(sub[col].value_counts())

reading: drive/MyDrive/medical_qa_hallucination/experiments/audit/audit_claims.csv
shape: (160, 12)
columns: ['uid', 'claim_id', 'claim_text', 'label', 'th_subtype', 'n_instances', 'arms_flagged', 'runs_flagged', 'cited_chunk', 'source_chunk_retrieved', 'outcome', 'note']
candidate type columns: ['th_subtype']

rows with a type label: 59
th_subtype
numeric_threshold         25
strength_logic            14
class_level                8
population_scope           6
fabricated_attribution     6
Name: count, dtype: int64


In [ ]:
# S3-MIXED: show rows whose fate is 'mixed'
import pandas as pd, re
from pathlib import Path
f = list(Path(".").rglob("step3_outcome_class_final.csv"))[0]
print("reading:", f)
df = pd.read_csv(f)
fcol = [c for c in df.columns if re.search("fate|outcome|class", c, re.I)]
print("candidate columns:", fcol)
c = fcol[0]
print(df[c].value_counts())
mix = df[df[c].astype(str).str.lower().str.contains("mixed")]
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print(mix.T)

reading: drive/MyDrive/medical_qa_hallucination/experiments/analysis/step3_outcome_class_final.csv
candidate columns: ['outcome_class']
outcome_class
corrected_clean    65
narrowed           36
correction_harm    19
resisted_kept      15
unresolved         15
mixed              10
Name: count, dtype: int64
                      8          36        59        70         83         85        101       103        111        149
uid            VHD_S12_Q1  AF_S9_Q11  AF_S6_Q5  AF_S9_Q8  CVD_S6_Q1  VHD_S5_Q1  AF_S3_Q3  AF_S5_Q1  AF_S9_Q12  AF_S10_Q2
claim_id                A          B         A         A          B          B         A         A          A          A
outcome_class       mixed      mixed     mixed     mixed      mixed      mixed     mixed     mixed      mixed      mixed


In [2]:
# S3-MIXED2: outcome classes broken down by audit label
import pandas as pd
from pathlib import Path
audit = pd.read_csv(list(Path(".").rglob("audit_claims.csv"))[0])
outc  = pd.read_csv(list(Path(".").rglob("step3_outcome_class_final.csv"))[0])
print("audit label counts:", audit['label'].value_counts().to_dict())
m = outc.merge(audit[['uid','claim_id','label']], on=['uid','claim_id'], how='left')
print("\noutcome_class x label:")
print(pd.crosstab(m['outcome_class'], m['label'], margins=True))
th = m[m['label'].astype(str).str.contains('TH', case=False, na=False)]
print("\nTH rows:", len(th))
print(th['outcome_class'].value_counts())
print("\nTH rows with mixed fate:")
print(th[th['outcome_class']=='mixed'][['uid','claim_id']])

audit label counts: {'verifier_error': 75, 'true_hallucination': 59, 'cross_source': 18, 'version_mismatch': 8}

outcome_class x label:
label            cross_source  true_hallucination  verifier_error  \
outcome_class                                                       
corrected_clean             3                  38              18   
correction_harm             0                   4              15   
mixed                       3                   2               5   
narrowed                   11                   8              17   
resisted_kept               1                   1              12   
unresolved                  0                   6               8   
All                        18                  59              75   

label            version_mismatch  All  
outcome_class                           
corrected_clean                 6   65  
correction_harm                 0   19  
mixed                           0   10  
narrowed                        0   3